# Image Captioning: results analysis and spatial attention

This notebook is the next part of the same project. **FinalAML.ipynb and all 79 completed global-vector runs stay unchanged.** All code needed here is included below; no Python helper file or execution of the old notebook is required. We can merge these sections into one submission notebook later.

The best validation configurations are ResNet50 + 3-layer LSTM and ResNet50 + 3-layer Transformer-prefix. They remain saved controls. The new experiments use 49 visual locations instead of one global vector: a 3-layer LSTM with additive spatial attention and a 3-layer Transformer with cross-attention. These are new architectures, trained from scratch with separate checkpoints.

Keep the saved split, train-only vocabulary, caption sampling, validation scorer, seed, batch size and training budget. Select models using validation; the labeled test stays unused. These are single-seed experiments, so small score differences do not establish significance.


## 1. Imports and settings

Start Jupyter in this project folder with the `deep-learning` kernel. All imports are here. Review mode does not start full training.

In [ ]:
import os

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import io
import json
import pickle
import random
import re
import time
import hashlib
import unicodedata
import textwrap
from pathlib import Path
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timezone
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps, ImageFile
import torch
import torch.nn as nn
import torchvision
from torch.utils.data import Dataset, DataLoader, Subset
from torch.nn.utils.rnn import pad_sequence
from torchvision import transforms
from torchvision.models import (
    resnet50,
    ResNet50_Weights,
    resnet18,
    ResNet18_Weights,
    vgg19,
    VGG19_Weights,
)
import csv
import inspect
import importlib.metadata as package_metadata
from functools import partial
from IPython.display import display, Markdown
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.bleu.bleu_scorer import BleuScorer
from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.cider.cider_scorer import CiderScorer

from torchvision.models import (
    mobilenet_v2,
    MobileNet_V2_Weights,
    efficientnet_v2_s,
    EfficientNet_V2_S_Weights,
)

import shutil
import tempfile
import zipfile

import gc
import math


In [ ]:
PROJECT_DIR = Path.cwd()
DATA_ROOT = PROJECT_DIR / "data" / "coco2017"
if not (DATA_ROOT / "annotations" / "captions_train2017.json").exists():
    DATA_ROOT = Path.home() / "Downloads" / "MSCOCO" / "coco2017"

# Review mode: enable extraction once, then enable training when ready.
EXTRACT_SPATIAL_FEATURES = False
RUN_PREFLIGHT = False
RUN_SPATIAL_TRAINING = False
SPATIAL_DECODERS = ["lstm_attention", "transformer_cross_attention"]
FEATURE_BATCH_SIZE = 64
TRAIN_BATCH_SIZE = 64
RESULTS_ROOT = PROJECT_DIR / "experiments"
GLOBAL_EXPERIMENT_DIR = RESULTS_ROOT / "all_encoders_v5_100epochs"
SPATIAL_EXPERIMENT_DIR = RESULTS_ROOT / "resnet50_spatial_v1"
SPATIAL_EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
CONTROL_RUNS = {
    "lstm_attention": "resnet50_lstm_depth_3_seed42",
    "transformer_cross_attention": "resnet50_transformer_depth_3_seed42",
}
FEATURE_DIMS = {
    "resnet50": 2048,
    "resnet18": 512,
    "vgg19": 4096,
    "mobilenet_v2": 1280,
    "efficientnet_v2_s": 1280,
}
SEED = 42
torch.set_num_threads(4)
ImageFile.LOAD_TRUNCATED_IMAGES = False
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", torch.cuda.get_device_name(0) if device.type == "cuda" else "CPU")
print("Full model training is", "ON" if RUN_SPATIAL_TRAINING else "OFF")

SPATIAL_IMPLEMENTATION_ID = "04025c8a9845faac540bd62a85603c7aef48c19600754be1864ac5c7d58b70f2"


The default is **Adam 0.001, batch 64, up to 100 epochs, patience 3 on validation CIDEr-D, dropout 0.3 and seed 42**. Training selects one random human caption per image per epoch; validation loss uses a fixed caption, while metrics retain all human references.

`EXTRACT_SPATIAL_FEATURES=True` creates or resumes the spatial cache; completed caches are reused. After it is ready, `RUN_SPATIAL_TRAINING=True` starts or resumes only the two new models. Keep the other settings fixed. If batch 64 does not fit your GPU, select a smaller batch before creating a run and use a separate result group for that changed protocol. Timers record active work and device details.


In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)


def device_details(device):
    return {
        "device": str(device),
        "name": torch.cuda.get_device_name(device) if device.type == "cuda" else "CPU",
        "torch": torch.__version__,
        "cuda_runtime": torch.version.cuda,
    }


def format_time(seconds):
    seconds = max(0, int(round(seconds)))
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"


def save_weights(path, model):
    # Plain weights for inference. Resume training with the full best.pt/last.pt checkpoint.
    path = Path(path)
    temporary = path.with_suffix(".pt.tmp")
    torch.save(model.state_dict(), temporary)
    temporary.replace(path)


def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding="utf-8")
    temporary.replace(path)


def rng_state():
    return {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
        "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    }


def restore_rng(state):
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])
    if torch.cuda.is_available() and state["cuda"] is not None:
        torch.cuda.set_rng_state_all(state["cuda"])


def experiment_identity(data, config, evaluator):
    return {
        "implementation": implementation_for(config),
        "config": config,
        "data": data["provenance"],
        "metrics": evaluator.metadata,
        "feature_dim": FEATURE_DIMS[config["encoder"]],
        "split_sizes": {name: len(values) for name, values in data["split_ids"].items()},
    }


def make_loader(dataset, batch_size, indices=None, seed=42):
    selected = dataset if indices is None else Subset(dataset, indices)
    # Its private generator does not consume the model's RNG when an iterator starts.
    return DataLoader(
        selected,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        collate_fn=partial(cached_caption_collate, pad_id=dataset.data["ids"]["pad"]),
        pin_memory=torch.cuda.is_available(),
        generator=torch.Generator().manual_seed(seed),
    )


def training_order(size, seed, epoch):
    return torch.randperm(size, generator=torch.Generator().manual_seed(seed + epoch)).tolist()


## 2. Analyze the 79 completed experiments

Scores come from saved validation results, not from the labeled test. Training totals include validation and checkpoint writing; stopping epochs and devices also affect them.

In [ ]:
def analyze_global_results():
    source = Path(GLOBAL_EXPERIMENT_DIR)
    output = Path(SPATIAL_EXPERIMENT_DIR) / "analysis"
    if output.resolve().is_relative_to(source.resolve()):
        raise ValueError(
            "Save analysis in the new spatial experiment group, outside the original runs."
        )

    plan = json.loads((source / "experiment_plan.json").read_text(encoding="utf-8"))
    planned = [item["config"] for item in plan["runs"]]
    assert len(planned) == len({config["run_name"] for config in planned}) == 79
    required = [
        "config.json",
        "summary.json",
        "history.json",
        "status.json",
        "vocabulary.json",
        "best.pt",
        "last.pt",
        "best_weights.pt",
        "last_weights.pt",
        "predictions/val_best.json",
    ]
    rows = []
    protocol = None
    for config in planned:
        name = config["run_name"]
        folder = source / name
        for filename in required:
            path = folder / filename
            assert path.is_file() and path.stat().st_size > 0, f"Missing or empty file: {path}"
        identity = json.loads((folder / "config.json").read_text(encoding="utf-8"))
        summary = json.loads((folder / "summary.json").read_text(encoding="utf-8"))
        history = json.loads((folder / "history.json").read_text(encoding="utf-8"))
        status = json.loads((folder / "status.json").read_text(encoding="utf-8"))
        assert identity["config"] == config, f"Configuration mismatch: {name}"
        assert identity["implementation"] == plan["implementation_by_decoder"][config["decoder"]]
        current_protocol = {key: identity[key] for key in ["data", "metrics", "split_sizes"]}
        if protocol is None:
            protocol = current_protocol
        assert current_protocol == protocol, f"Data or metric protocol mismatch: {name}"
        assert status["status"] == summary["status"] == "finished", f"Unfinished run: {name}"
        assert [record["epoch"] for record in history] == list(range(1, len(history) + 1))
        assert all(
            np.isfinite(record[key])
            for record in history
            for key in ["train_loss", "val_loss", "bleu1", "bleu4", "cider_d"]
        )
        best = max(history, key=lambda record: record["cider_d"])
        assert summary["best_epoch"] == best["epoch"], f"Best epoch mismatch: {name}"
        assert all(summary[key] == best[key] for key in ["val_loss", "bleu1", "bleu4", "cider_d"])
        best_score, bad_epochs = float("-inf"), 0
        for record in history:
            if record["cider_d"] > best_score:
                best_score, bad_epochs = record["cider_d"], 0
            else:
                bad_epochs += 1
            assert record["epoch"] == len(history) or bad_epochs < config["patience"]
        assert bad_epochs == config["patience"] == 3
        assert status["epoch"] == len(history) == best["epoch"] + 3 < config["max_epochs"]
        rows.append({"run": name, "config": config, "summary": summary, "history": history})

    winners = [
        max(
            [row for row in rows if row["config"]["decoder"] == decoder],
            key=lambda row: row["summary"]["cider_d"],
        )
        for decoder in ["rnn", "lstm", "transformer"]
    ]
    lines = [
        "| Decoder | Best run | CIDEr-D | BLEU-1 | BLEU-4 | Best / stop epoch | Time |",
        "|---|---|---:|---:|---:|---:|---|",
    ]
    for row in winners:
        score = row["summary"]
        lines.append(
            f"|{row['config']['decoder']}|{row['run']}|{score['cider_d']:.4f}|"
            f"{score['bleu1']:.4f}|{score['bleu4']:.4f}|"
            f"{score['best_epoch']} / {len(row['history'])}|{score['training_time']}|"
        )
    display(Markdown("### Best saved model in each decoder family\n\n" + "\n".join(lines)))

    depths = [
        row
        for row in rows
        if row["config"]["encoder"] == "resnet50"
        and row["config"]["decoder"] in ["lstm", "transformer"]
        and row["config"]["variant"] in ["baseline", "stacked", "depth_3", "depth_4"]
    ]
    depths.sort(key=lambda row: (row["config"]["decoder"], row["config"]["num_layers"]))
    assert len(depths) == 8
    lines = [
        "| Decoder | Layers | Parameters | Val loss | CIDEr-D | BLEU-1 | BLEU-4 | Best / stop | Time |",
        "|---|---:|---:|---:|---:|---:|---:|---:|---|",
    ]
    for row in depths:
        config, score = row["config"], row["summary"]
        lines.append(
            f"|{config['decoder']}|{config['num_layers']}|{score['trainable_parameters']:,}|"
            f"{score['val_loss']:.4f}|{score['cider_d']:.4f}|{score['bleu1']:.4f}|"
            f"{score['bleu4']:.4f}|{score['best_epoch']} / {len(row['history'])}|"
            f"{score['training_time']}|"
        )
    display(Markdown("### ResNet50: compare 1, 2, 3 and 4 layers\n\n" + "\n".join(lines)))

    output.mkdir(parents=True, exist_ok=True)
    fig, ax = plt.subplots(figsize=(7, 4))
    for decoder in ["lstm", "transformer"]:
        selected = [row for row in depths if row["config"]["decoder"] == decoder]
        ax.plot(
            [row["config"]["num_layers"] for row in selected],
            [row["summary"]["cider_d"] for row in selected],
            marker="o",
            label=decoder.upper(),
        )
    ax.set(
        xlabel="Number of layers", ylabel="Validation CIDEr-D", title="ResNet50 depth comparison"
    )
    ax.set_xticks([1, 2, 3, 4])
    ax.legend()
    ax.grid(alpha=0.2)
    fig.tight_layout()
    fig.savefig(output / "global_depth_cider.png", dpi=150, bbox_inches="tight")
    plt.show()

    fig, axes = plt.subplots(2, 3, figsize=(16, 8), squeeze=False)
    metrics = [
        ("train_loss", "Training loss"),
        ("val_loss", "Validation loss"),
        ("cider_d", "Validation CIDEr-D"),
    ]
    for index, decoder in enumerate(["lstm", "transformer"]):
        selected = [row for row in depths if row["config"]["decoder"] == decoder]
        for column, (metric, label) in enumerate(metrics):
            ax = axes[index, column]
            for row in selected:
                history, score = row["history"], row["summary"]
                (line,) = ax.plot(
                    [record["epoch"] for record in history],
                    [record[metric] for record in history],
                    label=f"{row['config']['num_layers']} layers",
                )
                if metric == "cider_d":
                    ax.scatter(score["best_epoch"], score[metric], color=line.get_color(), s=25)
            ax.set(title=f"{decoder.upper()}: {label}", xlabel="Epoch", ylabel=label)
            ax.legend()
            ax.grid(alpha=0.2)
    fig.tight_layout()
    fig.savefig(output / "global_depth_learning_curves.png", dpi=150, bbox_inches="tight")
    plt.show()

    findings = []
    deltas = {}
    for decoder in ["lstm", "transformer"]:
        selected = {
            row["config"]["num_layers"]: row["summary"]
            for row in depths
            if row["config"]["decoder"] == decoder
        }
        deltas[decoder] = {}
        for before, after in [(2, 3), (3, 4)]:
            change = {
                key: selected[after][key] - selected[before][key]
                for key in ["cider_d", "bleu1", "bleu4", "training_seconds"]
            }
            deltas[decoder][f"{after}_minus_{before}_layers"] = change
            findings.append(
                f"{decoder.upper()}, {after} vs {before} layers: "
                f"CIDEr-D {change['cider_d']:+.4f}, BLEU-4 {change['bleu4']:+.4f}."
            )
    devices = sorted({device["name"] for row in rows for device in row["summary"]["devices"]})
    findings.append("Recorded devices across the 79 runs: " + "; ".join(devices) + ".")
    findings.append(
        "Each configuration uses one seed. Small score differences do not establish a reliable advantage."
    )
    improving_loss = sum(
        row["history"][-1]["val_loss"] < row["summary"]["val_loss"] for row in depths
    )
    findings.append(
        f"In {improving_loss} of the 8 depth runs, validation loss was lower at stopping than at the best CIDEr-D epoch. Early stopping follows independently generated captions, so lower teacher-forced loss alone does not keep a run going."
    )
    findings.append(
        "These are the best depths under this fixed training and stopping protocol, not evidence that deeper models can never improve. The curves alone do not establish overfitting as the reason for the four-layer regressions."
    )
    findings.append(
        "Times include training, validation and checkpoint writing. Different devices and stopping epochs affect totals."
    )
    findings.append(
        "This checks saved JSON records and file presence. Metrics are not recomputed and checkpoint tensors are not loaded."
    )
    display(Markdown("### Findings\n\n" + "\n".join("- " + text for text in findings)))
    report = {
        "source_group": str(source),
        "verified_runs": len(rows),
        "protocol": protocol,
        "best_by_decoder": {row["config"]["decoder"]: row["run"] for row in winners},
        "depth_deltas": deltas,
        "findings": findings,
        "runs": rows,
    }
    (output / "global_results_analysis.json").write_text(
        json.dumps(report, indent=2), encoding="utf-8"
    )
    print(f"Loaded {len(rows)} completed global-vector runs. Analysis saved to: {output}")
    return rows


GLOBAL_RESULTS = analyze_global_results()


## 3. Load the existing preparation

Reuse the saved train/validation/test IDs, vocabulary and descriptions. The checks below reject image overlaps and exact duplicate photos across splits, verify the vocabulary uses only training captions, and keep original references for metrics.

In [ ]:
def load_prepared_data(project_dir, data_root=None):
    project_dir = Path(project_dir)
    data_root = Path(data_root) if data_root is not None else project_dir / "data" / "coco2017"
    for source in ["train2017", "val2017"]:
        annotation_path = data_root / "annotations" / f"captions_{source}.json"
        if not annotation_path.is_file():
            raise FileNotFoundError(
                f"Missing {annotation_path}. Extract the shared training-data bundle beside "
                "FinalAML.ipynb, or set DATA_ROOT to your existing COCO folder. "
                "Raw photos are not required for training from saved vectors."
            )
    cache = project_dir / "cache"
    required = [
        "splits.json",
        "prepared.pkl",
        "text_report.json",
        "vocab.json",
        "image_audit.json",
        "test_captions.json",
    ]
    missing = [name for name in required if not (cache / name).is_file()]
    if missing:
        raise FileNotFoundError(
            f"Missing prepared files: {missing}. Extract the shared training-data bundle into PROJECT_DIR. "
            "For a new dataset, run this notebook with TRAINING_ONLY=False to prepare it."
        )

    # Load saved preparation results, then verify their versions agree.
    split_bytes = (cache / "splits.json").read_bytes()
    vocab_bytes = (cache / "vocab.json").read_bytes()
    prepared_bytes = (cache / "prepared.pkl").read_bytes()
    split_file = json.loads(split_bytes)
    vocab = json.loads(vocab_bytes)
    report = json.loads((cache / "text_report.json").read_text(encoding="utf-8"))
    audit = json.loads((cache / "image_audit.json").read_text(encoding="utf-8"))

    prepared_sha = hashlib.sha256(prepared_bytes).hexdigest()
    assert (
        prepared_sha == report["cache_sha256"]
    ), "Prepared caption cache changed. Rerun preparation."
    prepared = pickle.loads(
        prepared_bytes
    )  # This is our own cache, checked against its saved hash.
    del prepared_bytes

    split_ids = split_file["ids"]
    sources = split_file["feature_sources"]
    assert set(split_ids) == {"train", "val", "test"}
    assert sources == {
        "train": "train2017",
        "val": "val2017",
        "test": "train2017",
    }, "Use the saved labeled split, not official test2017."
    payload = {"config": split_file["config"], "ids": split_ids, "feature_sources": sources}
    split_id = hashlib.sha256(
        json.dumps(payload, sort_keys=True, separators=(",", ":")).encode()
    ).hexdigest()
    assert split_id == split_file["split_id"] == prepared["config"]["split_id"] == vocab["split_id"]

    source_id = split_file["config"]["source_id"]
    assert source_id == prepared["config"]["source_id"] == vocab["source_id"] == audit["source_id"]
    assert prepared["config"] == report["config"]
    assert audit["complete"] and not audit["image_errors"] and not audit["metadata_errors"]
    assert (
        prepared["token_to_id"] == vocab["token_to_id"]
        and prepared["id_to_token"] == vocab["id_to_token"]
    )
    assert vocab["token_to_id"] == {word: index for index, word in enumerate(vocab["id_to_token"])}

    ids = {
        name: vocab["token_to_id"][token]
        for name, token in [
            ("pad", "<pad>"),
            ("unk", "<unk>"),
            ("start", "<start>"),
            ("end", "<end>"),
        ]
    }
    assert len(set(ids.values())) == 4

    # One photograph and its exact duplicates must stay in one labeled split.
    sets = {name: set(values) for name, values in split_ids.items()}
    for name, values in split_ids.items():
        assert values == sorted(sets[name]) and values, f"Invalid or repeated IDs in {name}."
    for left, right in [("train", "val"), ("train", "test"), ("val", "test")]:
        assert not sets[left] & sets[right], f"Image overlap: {left}/{right}."
    assert len(sets["test"]) == split_file["config"]["holdout_size"]

    owners = {
        (sources[name], image_id): name for name, values in sets.items() for image_id in values
    }
    for group in audit["pixel_duplicates"] + audit["file_duplicates"]:
        assigned = {owners[tuple(item)] for item in group if tuple(item) in owners}
        assert len(assigned) <= 1, "Exact duplicate photos cross the labeled splits."

    # Keep the original human references for evaluation.
    raw_by_source = {}
    source_image_ids = {}
    annotation_hashes = {}

    for source in ["train2017", "val2017"]:
        annotation_bytes = (data_root / "annotations" / f"captions_{source}.json").read_bytes()
        annotation_hashes[source] = hashlib.sha256(annotation_bytes).hexdigest()
        annotation = json.loads(annotation_bytes)
        source_image_ids[source] = sorted(row["id"] for row in annotation["images"])
        assert len(source_image_ids[source]) == len(set(source_image_ids[source]))

        raw_by_source[source] = defaultdict(list)
        for row in annotation["annotations"]:
            assert (
                isinstance(row["caption"], str) and row["caption"].strip()
            ), "Empty human reference."
            raw_by_source[source][row["image_id"]].append(row["caption"])
        assert set(raw_by_source[source]) == set(
            source_image_ids[source]
        ), "Missing human descriptions."

        if source == "train2017":
            saved_test = json.loads((cache / "test_captions.json").read_text(encoding="utf-8"))
            assert saved_test["split_id"] == split_id
            expected_test = [
                row for row in annotation["annotations"] if row["image_id"] in sets["test"]
            ]
            assert (
                saved_test["annotations"] == expected_test
            ), "Saved test references differ from the original human captions."
        del annotation_bytes, annotation

    assert sets["train"] | sets["test"] == set(source_image_ids["train2017"])
    assert sets["val"] == set(source_image_ids["val2017"])

    references = {
        name: {image_id: raw_by_source[sources[name]][image_id] for image_id in values}
        for name, values in split_ids.items()
    }
    tokens = {name: prepared[f"{name}_tokens"] for name in split_ids}
    for name, values in tokens.items():
        assert sorted(values) == split_ids[name]
        assert all(
            rows and all(row for row in rows) for rows in values.values()
        ), "Empty tokenized caption."
        if name != "train":
            assert all(
                len(rows) == len(references[name][image_id]) for image_id, rows in values.items()
            ), "Evaluation references were dropped."

    # Test and validation captions must not contribute to the vocabulary.
    train_counts = Counter(
        token for rows in tokens["train"].values() for row in rows for token in row
    )
    assert train_counts == prepared["train_counts"]
    settings = prepared["config"]["settings"]
    regular = [
        token
        for token, count in sorted(train_counts.items(), key=lambda row: (-row[1], row[0]))
        if count >= settings["min_frequency"]
    ][: settings["max_vocab_tokens"]]
    assert (
        vocab["id_to_token"] == settings["special_tokens"] + regular
    ), "Vocabulary must be fitted on training captions only."

    data = {
        "project_dir": project_dir,
        "data_root": data_root,
        "split_ids": split_ids,
        "feature_sources": sources,
        "tokens": tokens,
        "references": references,
        "source_image_ids": source_image_ids,
        "token_to_id": vocab["token_to_id"],
        "id_to_token": vocab["id_to_token"],
        "ids": ids,
        "max_content_tokens": prepared["config"]["max_content_tokens"],
        "text_settings": settings,
        "provenance": {
            "source_id": source_id,
            "split_id": split_id,
            "prepared_sha256": prepared_sha,
            "vocab_sha256": hashlib.sha256(vocab_bytes).hexdigest(),
            "splits_sha256": hashlib.sha256(split_bytes).hexdigest(),
            "annotations_sha256": annotation_hashes,
        },
    }

    data["feature_files"] = {}
    for encoder in FEATURE_DIMS:
        data["feature_files"][encoder] = validate_feature_files(data, encoder)
    data["provenance"]["feature_cache_keys"] = {
        encoder: {source: entry["cache_key"] for source, entry in files.items()}
        for encoder, files in data["feature_files"].items()
    }

    print(
        "Verified disjoint labeled splits:",
        {name: len(values) for name, values in split_ids.items()},
    )
    print("Vocabulary:", len(data["id_to_token"]), "| special IDs:", ids)
    return data


def validate_feature_files(data, encoder):
    if encoder not in FEATURE_DIMS:
        raise ValueError("Choose an encoder listed in FEATURE_DIMS.")

    directory = data["project_dir"] / "cache" / "features"
    if encoder != "resnet50":
        directory = directory / encoder

    files = {}
    for source in sorted(set(data["feature_sources"].values())):
        path = directory / f"{source}.npy"
        meta = json.loads((directory / f"{source}_meta.json").read_text(encoding="utf-8"))
        image_ids = json.loads((directory / f"{source}_image_ids.json").read_text(encoding="utf-8"))
        expected_shape = [len(data["source_image_ids"][source]), FEATURE_DIMS[encoder]]

        assert meta["complete"] and meta["config"]["source_id"] == data["provenance"]["source_id"]
        assert (
            hashlib.sha256(json.dumps(meta["config"], sort_keys=True).encode()).hexdigest()
            == meta["cache_key"]
        )
        assert image_ids == data["source_image_ids"][source] and meta["shape"] == expected_shape
        assert meta["dtype"] == "float32" and path.stat().st_size == meta["feature_file_bytes"]

        # Read the array header without loading all vectors into RAM.
        matrix = np.load(path, mmap_mode="r", allow_pickle=False)
        try:
            assert list(matrix.shape) == expected_shape and matrix.dtype == np.float32
        finally:
            matrix._mmap.close()

        files[source] = {
            "path": path,
            "image_ids": image_ids,
            "cache_key": meta["cache_key"],
            "shape": expected_shape,
        }
    return files


class CachedCaptionDataset(Dataset):
    def __init__(self, data, encoder, split, seed=42):
        if split not in ["train", "val", "test"]:
            raise ValueError("Use the project's train, val or labeled test.")

        self.data = data
        self.encoder = encoder
        self.split = split
        self.seed = seed
        self.image_ids = data["split_ids"][split]
        self.feature_dim = FEATURE_DIMS[encoder]

        source = data["feature_sources"][split]
        saved = data["feature_files"][encoder][source]
        self.features = np.load(saved["path"], mmap_mode="r", allow_pickle=False)
        self.feature_rows = {image_id: index for index, image_id in enumerate(saved["image_ids"])}
        self.set_epoch(0)

    def set_epoch(self, epoch):
        self.epoch = int(epoch)

        # A private RNG keeps caption selection independent of model dropout.
        rng = random.Random(self.seed + self.epoch)
        self.caption_choices = [
            (
                rng.randrange(len(self.data["tokens"][self.split][image_id]))
                if self.split == "train"
                else 0
            )
            for image_id in self.image_ids
        ]

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, index):
        image_id = self.image_ids[index]
        tokens = self.data["tokens"][self.split][image_id][self.caption_choices[index]]

        ids = self.data["ids"]
        caption = (
            [ids["start"]]
            + [
                self.data["token_to_id"].get(token, ids["unk"])
                for token in tokens[: self.data["max_content_tokens"]]
            ]
            + [ids["end"]]
        )

        # Float16 disk storage becomes float32 input; AMP controls training compute.
        vector = torch.from_numpy(self.features[self.feature_rows[image_id]].copy()).float()
        return {
            "features": vector,
            "caption": torch.tensor(caption, dtype=torch.long),
            "image_id": image_id,
            "references": self.data["references"][self.split][image_id],
        }

    def close(self):
        if self.features is not None:
            self.features._mmap.close()
            self.features = None


def cached_caption_collate(batch, pad_id):
    captions = [item["caption"] for item in batch]

    return {
        "features": torch.stack([item["features"] for item in batch]),
        "captions": pad_sequence(captions, batch_first=True, padding_value=pad_id),
        "lengths": torch.tensor([len(caption) for caption in captions], dtype=torch.long),
        "image_ids": [item["image_id"] for item in batch],
        "references": [item["references"] for item in batch],
    }


In [ ]:
data = load_prepared_data(PROJECT_DIR, DATA_ROOT)


## 4. Cache ResNet50 spatial features

For each padded 224 x 224 image, frozen ResNet50 produces `[2048,7,7]` after layer4 and before global pooling. Flattening rows then columns gives **`[49,2048]`**. The trainable 2048-to-512 projection belongs to the decoder, so it is not cached.

Old global vectors cannot recover these locations: averaging has removed their spatial arrangement. The new cache uses the same pretrained CNN, resize + padding, normalization and CUDA precision as the old cache. Float16 storage needs about **23 GiB** for train2017 + val2017. Those source images include the already separated labeled test; feature extraction uses no test captions or metrics. Official unlabeled test2017 is excluded.

The cache has its own folder, image-ID order, metadata and saved progress. Each committed batch is flushed before progress advances; an interrupted batch can be safely repeated. No old global cache is overwritten. Comparisons of spatial means with old global vectors check the preprocessing/order, allowing small floating-point rounding differences. All 49 positions remain available, including borders affected by image padding.


In [ ]:
SPATIAL_CACHE_NAME = "resnet50_spatial_v1"
SPATIAL_SHAPE = (49, 2048)
SPATIAL_DTYPE = np.dtype("float16")
SPATIAL_MEAN_RTOL = 0.005
SPATIAL_MEAN_ATOL = 0.005
SPATIAL_IMAGE_WORKERS = 8


def spatial_cache_directory(data):
    return Path(data["project_dir"]) / "cache" / "features" / SPATIAL_CACHE_NAME


def spatial_cache_spec(data):
    """Reuse the existing frozen ResNet50 weights, pixel preparation and precision."""
    global_dir = Path(data["project_dir"]) / "cache" / "features"
    global_meta = {
        source: json.loads((global_dir / f"{source}_meta.json").read_text(encoding="utf-8"))
        for source in ["train2017", "val2017"]
    }
    reference = global_meta["train2017"]
    keys = ["weights", "image_size", "resize", "padding_color", "mean", "std", "compute_precision"]
    for source, meta in global_meta.items():
        assert meta["complete"] and meta["config"]["source_id"] == data["provenance"]["source_id"]
        assert all(meta["config"][key] == reference["config"][key] for key in keys)
        assert meta["weights_sha256"] == reference["weights_sha256"]
        ids = json.loads((global_dir / f"{source}_image_ids.json").read_text(encoding="utf-8"))
        assert ids == data["source_image_ids"][source], "Global feature row order changed."
    original = reference["config"]
    assert original["weights"] == "ResNet50_Weights.IMAGENET1K_V2"
    assert original["image_size"] == 224
    assert original["resize"] == "ImageOps.pad,BILINEAR,centering=(0.5,0.5),EXIF transpose,RGB"
    assert (
        original["compute_precision"] == "CUDA autocast float16"
    ), "This spatial cache matches the CUDA global cache."
    return {
        "version": 1,
        "source_id": data["provenance"]["source_id"],
        "encoder": "resnet50",
        "weights": original["weights"],
        "weights_sha256": reference["weights_sha256"],
        "layer": "layer4 output, before avgpool and fc",
        "shape_per_image": list(SPATIAL_SHAPE),
        "tensor_order": "NCHW -> flatten H,W in row-major order -> N,49,2048",
        "storage_dtype": "float16",
        "cnn_frozen": True,
        "cnn_eval": True,
        **{key: original[key] for key in keys if key != "weights"},
        "global_cache_keys": {source: meta["cache_key"] for source, meta in global_meta.items()},
        "mean_check": {"rtol": SPATIAL_MEAN_RTOL, "atol": SPATIAL_MEAN_ATOL},
    }


def _spatial_key(spec):
    return hashlib.sha256(json.dumps(spec, sort_keys=True).encode()).hexdigest()


def _spatial_write_json(path, value):
    """A reader sees either the previous complete JSON or the next complete JSON."""
    temporary = path.with_suffix(".tmp")
    with temporary.open("w", encoding="utf-8") as handle:
        json.dump(value, handle, indent=2)
        handle.flush()
        os.fsync(handle.fileno())
    for attempt in range(10):
        try:
            temporary.replace(path)
            return
        except PermissionError:
            if attempt == 9:
                raise
            time.sleep(0.2)


def _spatial_paths(directory, source):
    return {
        "array": directory / f"{source}.npy",
        "partial": directory / f"{source}.partial.npy",
        "ids": directory / f"{source}_image_ids.json",
        "meta": directory / f"{source}_meta.json",
        "progress": directory / f"{source}_progress.json",
    }


def _spatial_check_header(path, shape):
    matrix = np.load(path, mmap_mode="r", allow_pickle=False)
    try:
        assert list(matrix.shape) == shape and matrix.dtype == SPATIAL_DTYPE
        expected_bytes = matrix.offset + int(np.prod(shape)) * SPATIAL_DTYPE.itemsize
        assert path.stat().st_size == expected_bytes, "Incomplete or unexpected array size."
    finally:
        matrix._mmap.close()


def _spatial_state(data, source, spec):
    """Never overwrite an incompatible cache or guess how many rows were committed."""
    directory = spatial_cache_directory(data)
    paths = _spatial_paths(directory, source)
    ids = data["source_image_ids"][source]
    shape = [len(ids), *SPATIAL_SHAPE]
    key = _spatial_key(spec)
    if paths["ids"].exists():
        assert (
            json.loads(paths["ids"].read_text(encoding="utf-8")) == ids
        ), "Spatial image IDs differ."
    if paths["meta"].exists():
        meta = json.loads(paths["meta"].read_text(encoding="utf-8"))
        assert meta["complete"] and meta["config"] == spec and meta["cache_key"] == key
        assert meta["source"] == source and meta["shape"] == shape and meta["dtype"] == "float16"
        assert paths["array"].exists() and paths["ids"].exists() and not paths["partial"].exists()
        _spatial_check_header(paths["array"], shape)
        assert paths["array"].stat().st_size == meta["feature_file_bytes"]
        return {"status": "complete", "processed": len(ids), "meta": meta, "paths": paths}
    if paths["progress"].exists():
        progress = json.loads(paths["progress"].read_text(encoding="utf-8"))
        assert (
            progress["config"] == spec
            and progress["cache_key"] == key
            and progress["source"] == source
        )
        assert (
            progress["shape"] == shape and progress["dtype"] == "float16" and paths["ids"].exists()
        )
        count = progress["processed"]
        assert isinstance(count, int) and not isinstance(count, bool) and 0 <= count <= len(ids)
        assert np.isfinite(progress["elapsed_seconds"]) and progress["elapsed_seconds"] >= 0
        assert progress["status"] in ["running", "paused", "ready"]
        if paths["array"].exists():
            # Recovery after array rename but before completion metadata was written.
            assert (
                progress["status"] == "ready"
                and count == len(ids)
                and not paths["partial"].exists()
            )
            _spatial_check_header(paths["array"], shape)
            return {"status": "ready", "processed": count, "progress": progress, "paths": paths}
        assert paths["partial"].exists(), "Checkpoint has no matching partial array."
        _spatial_check_header(paths["partial"], shape)
        return {"status": "partial", "processed": count, "progress": progress, "paths": paths}
    if paths["array"].exists() or paths["partial"].exists():
        raise ValueError(
            f"Uncommitted spatial file for {source}. Review {directory}; no existing file will be overwritten."
        )
    return {"status": "new", "processed": 0, "paths": paths}


def spatial_cache_plan(data, reserve_gib=2):
    """Read-only: estimate space and show which original splits need extraction."""
    spec = spatial_cache_spec(data)
    states = {source: _spatial_state(data, source, spec) for source in ["train2017", "val2017"]}
    bytes_per_image = int(np.prod(SPATIAL_SHAPE)) * SPATIAL_DTYPE.itemsize
    total_bytes = sum(
        len(data["source_image_ids"][source]) * bytes_per_image + 128 for source in states
    )
    remaining_bytes = sum(
        (len(data["source_image_ids"][source]) - state["processed"]) * bytes_per_image
        for source, state in states.items()
        if state["status"] not in ["complete", "ready"]
    )
    additional_bytes = 0
    allocation_notes = {}
    for source, state in states.items():
        expected_bytes = len(data["source_image_ids"][source]) * bytes_per_image + 128
        if state["status"] == "new":
            extra = expected_bytes
            note = "New array: reserve its full file size."
        elif state["status"] == "partial":
            info = state["paths"]["partial"].stat()
            if hasattr(info, "st_blocks"):
                extra = max(0, expected_bytes - info.st_blocks * 512)
                note = "Use the difference between logical size and allocated filesystem blocks."
            elif (
                os.name == "nt"
                and hasattr(info, "st_file_attributes")
                and not info.st_file_attributes & (0x200 | 0x800)
            ):
                # 0x200 is SPARSE_FILE; 0x800 is COMPRESSED. Normal Windows arrays
                # already reserve their full validated size when open_memmap creates them.
                extra = max(0, expected_bytes - info.st_size)
                note = "Ordinary full-size Windows file: its storage is already reserved."
            else:
                extra = (
                    len(data["source_image_ids"][source]) - state["processed"]
                ) * bytes_per_image
                note = "Sparse/compressed or unknown allocation: conservatively reserve uncommitted rows."
        else:
            extra = 0
            note = "Complete or ready array: no additional allocation."
        additional_bytes += extra
        allocation_notes[source] = {"additional_bytes": extra, "reason": note}
    free_bytes = shutil.disk_usage(Path(data["project_dir"])).free
    reserve_bytes = int(reserve_gib * 1024**3)
    required_free = additional_bytes + reserve_bytes if remaining_bytes else 0
    return {
        "cache_dir": str(spatial_cache_directory(data)),
        "shape_per_image": list(SPATIAL_SHAPE),
        "dtype": "float16",
        "total_bytes": total_bytes,
        "total_gib": total_bytes / 1024**3,
        "remaining_bytes": remaining_bytes,
        "free_bytes": free_bytes,
        "additional_allocation_bytes": additional_bytes,
        "allocation_notes": allocation_notes,
        "required_free_bytes": required_free,
        "enough_space": free_bytes >= required_free,
        "sources": {
            source: {
                "status": state["status"],
                "processed": state["processed"],
                "total": len(data["source_image_ids"][source]),
            }
            for source, state in states.items()
        },
        "space_note": "Free-space check counts new allocation plus reserve, not already reserved ordinary Windows partial files.",
    }


def load_spatial_feature_files(data, require_complete=True):
    """Read headers/metadata only; never load the complete spatial array into RAM."""
    spec = spatial_cache_spec(data)
    files = {}
    for source in ["train2017", "val2017"]:
        state = _spatial_state(data, source, spec)
        if state["status"] != "complete":
            if require_complete:
                raise FileNotFoundError(
                    f"Spatial {source} cache is {state['status']}. Run the explicit extraction cell first."
                )
            continue
        files[source] = {
            "path": state["paths"]["array"],
            "image_ids": data["source_image_ids"][source],
            "cache_key": _spatial_key(spec),
            "shape": state["meta"]["shape"],
            "dtype": "float16",
            "meta": state["meta"],
        }
    return files


def _spatial_image_records(data, source, photo_root):
    annotation = json.loads(
        (Path(data["data_root"]) / "annotations" / f"captions_{source}.json").read_text(
            encoding="utf-8"
        )
    )
    by_id = {row["id"]: row for row in annotation["images"]}
    assert sorted(by_id) == data["source_image_ids"][source]
    records = [by_id[image_id] for image_id in data["source_image_ids"][source]]
    directory = Path(photo_root) / source
    if not directory.is_dir():
        raise FileNotFoundError(f"Spatial extraction needs the original JPEGs: {directory}")
    available = {entry.name for entry in os.scandir(directory) if entry.is_file()}
    missing = [row["file_name"] for row in records if row["file_name"] not in available]
    if missing:
        raise FileNotFoundError(f"{source}: {len(missing)} photos missing; first: {missing[:3]}")
    assert all(Path(row["file_name"]).name == row["file_name"] for row in records)
    return records


def resolve_spatial_photo_root(data, photo_root=None):
    """Training bundles can contain annotations without the original JPEG folders."""
    candidates = (
        [Path(photo_root)]
        if photo_root is not None
        else [
            Path(data["data_root"]),
            Path(data["project_dir"]) / "data" / "coco2017",
            Path.home() / "Downloads" / "MSCOCO" / "coco2017",
        ]
    )
    failures = []
    checked = set()
    for candidate in candidates:
        if str(candidate.resolve()) in checked:
            continue
        checked.add(str(candidate.resolve()))
        if not all((candidate / source).is_dir() for source in ["train2017", "val2017"]):
            failures.append(f"{candidate}: train2017/val2017 JPEG folders are missing")
            continue
        try:
            for source in ["train2017", "val2017"]:
                _spatial_image_records(data, source, candidate)
            return candidate
        except FileNotFoundError as error:
            failures.append(str(error))
    raise FileNotFoundError(
        "Original photos are required for spatial extraction. Set photo_root to the full COCO folder. "
        + "; ".join(failures)
    )


class SpatialImageDataset(Dataset):
    def __init__(self, photo_root, source, records, spec, start=0):
        self.directory = Path(photo_root) / source
        self.records = records[start:]
        self.spec = spec
        self.transform = transforms.Compose(
            [transforms.ToTensor(), transforms.Normalize(spec["mean"], spec["std"])]
        )

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        with Image.open(self.directory / self.records[index]["file_name"]) as image:
            image = ImageOps.exif_transpose(image).convert("RGB")
            image = ImageOps.pad(
                image,
                (224, 224),
                method=Image.Resampling.BILINEAR,
                color=tuple(self.spec["padding_color"]),
                centering=(0.5, 0.5),
            )
            return self.transform(image)


def _spatial_image_batches(dataset, batch_size, pin_memory=False, workers=SPATIAL_IMAGE_WORKERS):
    """Threads decode the next ordered batch; no Windows worker processes are needed."""
    with ThreadPoolExecutor(max_workers=workers) as pool:
        for start in range(0, len(dataset), batch_size):
            indices = range(start, min(start + batch_size, len(dataset)))
            images = list(pool.map(dataset.__getitem__, indices))
            batch = torch.stack(images)
            yield batch.pin_memory() if pin_memory else batch


def _spatial_mean_check(regions, global_vectors):
    pooled = np.asarray(regions, dtype=np.float32).mean(axis=1)
    expected = np.asarray(global_vectors, dtype=np.float32)
    difference = pooled - expected
    check = {
        "images": len(pooled),
        "max_absolute_error": float(np.abs(difference).max()),
        "relative_l2_error": float(
            np.linalg.norm(difference) / max(np.linalg.norm(expected), 1e-12)
        ),
        "rtol": SPATIAL_MEAN_RTOL,
        "atol": SPATIAL_MEAN_ATOL,
    }
    if not np.allclose(pooled, expected, rtol=SPATIAL_MEAN_RTOL, atol=SPATIAL_MEAN_ATOL):
        raise ValueError(f"Spatial mean does not match the saved global vectors: {check}")
    check["passed"] = True
    return check


def validate_spatial_means(data):
    files = load_spatial_feature_files(data)
    checks = {}
    for source, saved in files.items():
        spatial = np.load(saved["path"], mmap_mode="r", allow_pickle=False)
        global_vectors = np.load(
            Path(data["project_dir"]) / "cache" / "features" / f"{source}.npy",
            mmap_mode="r",
            allow_pickle=False,
        )
        try:
            rows = sorted({0, len(spatial) // 2, len(spatial) - 1})
            checks[source] = {
                "rows": rows,
                **_spatial_mean_check(spatial[rows], global_vectors[rows]),
            }
        finally:
            spatial._mmap.close()
            global_vectors._mmap.close()
    return checks


def _spatial_complete_metadata(paths, progress, spec, source):
    if paths["partial"].exists():
        paths["partial"].replace(paths["array"])
    _spatial_check_header(paths["array"], progress["shape"])
    meta = {
        **progress,
        "status": "complete",
        "complete": True,
        "source": source,
        "config": spec,
        "feature_file_bytes": paths["array"].stat().st_size,
        "completed_utc": datetime.now(timezone.utc).isoformat(),
    }
    _spatial_write_json(paths["meta"], meta)
    return meta


def _extract_spatial_source(data, source, records, photo_root, encoder, device, spec, batch_size):
    state = _spatial_state(data, source, spec)
    if state["status"] == "complete":
        return state["meta"]
    paths = state["paths"]
    if state["status"] == "ready":
        return _spatial_complete_metadata(paths, state["progress"], spec, source)
    ids = data["source_image_ids"][source]
    offset = state["processed"]
    old_progress = state.get("progress", {})
    previous_seconds = float(old_progress.get("elapsed_seconds", 0))
    first_started = old_progress.get("started_utc", datetime.now(timezone.utc).isoformat())
    checks = old_progress.get("mean_checks", [])
    resumed_from = offset
    _spatial_write_json(paths["ids"], ids)
    dataset = SpatialImageDataset(photo_root, source, records, spec, start=offset)
    batches = _spatial_image_batches(dataset, batch_size, pin_memory=device.type == "cuda")
    if device.type == "cuda":
        torch.cuda.synchronize()
    start = time.perf_counter()
    matrix = (
        np.load(paths["partial"], mmap_mode="r+", allow_pickle=False)
        if state["status"] == "partial"
        else np.lib.format.open_memmap(
            paths["partial"], mode="w+", dtype=SPATIAL_DTYPE, shape=(len(ids), *SPATIAL_SHAPE)
        )
    )
    global_vectors = np.load(
        Path(data["project_dir"]) / "cache" / "features" / f"{source}.npy",
        mmap_mode="r",
        allow_pickle=False,
    )

    def save_progress(status):
        # Flush vectors BEFORE advancing the durable committed-row counter.
        matrix.flush()
        elapsed = previous_seconds + time.perf_counter() - start
        progress = {
            "schema_version": 1,
            "status": status,
            "complete": False,
            "source": source,
            "cache_key": _spatial_key(spec),
            "config": spec,
            "processed": offset,
            "total": len(ids),
            "shape": [len(ids), *SPATIAL_SHAPE],
            "dtype": "float16",
            "elapsed_seconds": elapsed,
            "started_utc": first_started,
            "updated_utc": datetime.now(timezone.utc).isoformat(),
            "resumed_from_row": resumed_from,
            "batch_size": batch_size,
            "device": str(device),
            "gpu": torch.cuda.get_device_name(device) if device.type == "cuda" else None,
            "producer": {
                "torch": torch.__version__,
                "torchvision": torchvision.__version__,
                "image_workers": SPATIAL_IMAGE_WORKERS,
            },
            "mean_checks": checks,
            "timing": "Active image reading, padding, normalization, inference, validation and saving; pauses, model loading and warm-up excluded.",
        }
        _spatial_write_json(paths["progress"], progress)
        return progress

    try:
        save_progress("running")
        last_save = time.perf_counter()
        checked_this_run = False
        with torch.inference_mode():
            for image_batch in batches:
                with torch.autocast(
                    device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"
                ):
                    grid = encoder(image_batch.to(device, non_blocking=True))
                assert grid.shape[1:] == (2048, 7, 7)
                regions = grid.flatten(2).transpose(1, 2).contiguous()
                assert torch.isfinite(
                    regions
                ).all(), f"Non-finite spatial features at {source}/{offset}."
                values = regions.to(dtype=torch.float16).cpu().numpy()
                assert np.isfinite(values).all() and np.any(values != 0, axis=(1, 2)).all()
                if not checked_this_run:
                    count = min(8, len(values))
                    checks.append(
                        {
                            "first_row": offset,
                            **_spatial_mean_check(
                                values[:count], global_vectors[offset : offset + count]
                            ),
                        }
                    )
                    checked_this_run = True
                matrix[offset : offset + len(values)] = values
                offset += len(values)
                if time.perf_counter() - last_save >= 20 or offset == len(ids):
                    progress = save_progress("running")
                    print(
                        f"Spatial {source}: {offset:,}/{len(ids):,}, active {progress['elapsed_seconds']/60:.2f} min",
                        flush=True,
                    )
                    last_save = time.perf_counter()
        assert offset == len(ids)
        progress = save_progress("ready")
    except BaseException:
        save_progress("paused")
        raise
    finally:
        matrix._mmap.close()
        global_vectors._mmap.close()
    return _spatial_complete_metadata(paths, progress, spec, source)


def extract_resnet50_spatial(data, device, batch_size=64, photo_root=None, resume=True):
    """Explicitly run once; completed sources are reused and partial sources resume."""
    device = torch.device(device)
    if device.type != "cuda":
        raise ValueError(
            "Use CUDA for production spatial extraction to match the saved global features' autocast precision."
        )
    assert isinstance(batch_size, int) and batch_size > 0
    spec = spatial_cache_spec(data)
    plan = spatial_cache_plan(data)
    print(json.dumps(plan, indent=2), flush=True)
    if not plan["enough_space"]:
        raise OSError(
            f"Need about {plan['required_free_bytes']/1024**3:.2f} GiB free including reserve; have {plan['free_bytes']/1024**3:.2f} GiB."
        )
    unfinished = [source for source, row in plan["sources"].items() if row["status"] != "complete"]
    if not resume and any(plan["sources"][source]["status"] != "new" for source in unfinished):
        raise ValueError(
            "A partial spatial cache exists. Set resume=True; completed or partial arrays are never overwritten."
        )
    if not unfinished:
        return {
            "files": load_spatial_feature_files(data),
            "mean_validation": validate_spatial_means(data),
        }
    photo_root = resolve_spatial_photo_root(data, photo_root)
    records = {source: _spatial_image_records(data, source, photo_root) for source in unfinished}
    weight_name = "resnet50-11ad3fa6.pth"
    candidates = [
        Path(data["project_dir"]) / "cache" / "models" / "checkpoints" / weight_name,
        Path(torch.hub.get_dir()) / "checkpoints" / weight_name,
        Path.home() / ".cache" / "torch" / "hub" / "checkpoints" / weight_name,
    ]
    weight_path = next((path for path in candidates if path.is_file()), None)
    if weight_path is None:
        raise FileNotFoundError(f"The already-used pretrained weights are missing: {weight_name}")
    assert (
        hashlib.sha256(weight_path.read_bytes()).hexdigest() == spec["weights_sha256"]
    ), "Pretrained weights differ from the global cache."
    directory = spatial_cache_directory(data)
    directory.mkdir(parents=True, exist_ok=True)
    old_threads = torch.get_num_threads()
    encoder = None
    try:
        torch.set_num_threads(1)
        backbone = resnet50(weights=None)
        backbone.load_state_dict(torch.load(weight_path, map_location="cpu", weights_only=True))
        encoder = (
            nn.Sequential(*list(backbone.children())[:-2]).requires_grad_(False).to(device).eval()
        )
        del backbone
        with torch.inference_mode(), torch.autocast(device_type="cuda", dtype=torch.float16):
            warmup = encoder(torch.zeros(2, 3, 224, 224, device=device))
            assert warmup.shape == (2, 2048, 7, 7) and torch.isfinite(warmup).all()
        del warmup
        torch.cuda.synchronize()
        for source in unfinished:
            _extract_spatial_source(
                data, source, records[source], photo_root, encoder, device, spec, batch_size
            )
    finally:
        del encoder
        torch.set_num_threads(old_threads)
        torch.cuda.empty_cache()
    files = load_spatial_feature_files(data)
    mean_validation = validate_spatial_means(data)
    report = {
        "complete": True,
        "cache_key": _spatial_key(spec),
        "config": spec,
        "total_images": sum(len(item["image_ids"]) for item in files.values()),
        "total_active_seconds": sum(item["meta"]["elapsed_seconds"] for item in files.values()),
        "splits": {source: item["meta"] for source, item in files.items()},
        "mean_validation": mean_validation,
    }
    _spatial_write_json(directory / "feature_report.json", report)
    print(
        f"Spatial cache complete: {report['total_images']:,} images, {report['total_active_seconds']/60:.2f} active min."
    )
    return {"files": files, "mean_validation": mean_validation, "report": report}


In [ ]:
print(spatial_cache_plan(data))
if EXTRACT_SPATIAL_FEATURES:
    spatial_extraction = extract_resnet50_spatial(
        data, device, batch_size=FEATURE_BATCH_SIZE, resume=True
    )
else:
    print("Extraction is off; completed spatial features will be reused below.")
completed_feature_files = load_spatial_feature_files(data, require_complete=False)
for source, saved in completed_feature_files.items():
    meta = saved["meta"]
    print(
        source,
        "| shape:",
        saved["shape"],
        "| saved extraction time:",
        format_time(meta["elapsed_seconds"]),
        "| device:",
        meta["gpu"],
    )
if len(completed_feature_files) == 2:
    print(
        "Total saved active extraction time:",
        format_time(
            sum(saved["meta"]["elapsed_seconds"] for saved in completed_feature_files.values())
        ),
    )


## 5. Spatial decoders

**LSTM:** project 49 locations to 512. Starting from zero hidden/cell states, process their mean as one visual step. At each word, additive attention combines the previous top-layer hidden state with all locations. Concatenate the attended context with the word embedding, project it back to 512, and update all three LSTM layers. The visual output has no word target. Built-in recurrent dropout stays zero; separate dropout follows embeddings and precedes the vocabulary output.

**Transformer:** words are `tgt`; the projected 49 locations are `memory`. Three independent `TransformerDecoderLayer` blocks have width 512, 8 heads, feed-forward width 2048 and pre-normalization. Text has learned positions and a causal mask. Memory has learned row/column positions and no causal mask; every word may read every visual location. This differs from the old image-prefix architecture, so its old checkpoints are not loaded into this model.

The LSTM design is an additive-attention adaptation, not an exact reproduction of [Show, Attend and Tell](https://arxiv.org/html/1502.03044v3). See the [PyTorch LSTM](https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html) and [TransformerDecoderLayer](https://docs.pytorch.org/docs/stable/generated/torch.nn.TransformerDecoderLayer.html) APIs.


In [ ]:
class CaptionSpatialLSTM(nn.Module):
    def __init__(
        self,
        feature_dim,
        vocab_size,
        pad_id,
        embedding_size=512,
        hidden_size=512,
        num_layers=3,
        dropout=0.3,
        grid_size=7,
    ):
        super().__init__()
        if embedding_size != hidden_size:
            raise ValueError("This model uses equal embedding and hidden sizes.")
        if num_layers < 1 or grid_size < 1:
            raise ValueError("Use at least one layer and a positive grid size.")

        self.decoder_type = "lstm_attention"
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.pad_id = pad_id
        self.grid_size = grid_size

        self.image_projection = nn.Linear(feature_dim, hidden_size)
        self.embedding = nn.Embedding(vocab_size, embedding_size, padding_idx=pad_id)
        self.embedding_dropout = nn.Dropout(dropout)

        # Additive attention combines each image location with the previous hidden state.
        self.attention_visual = nn.Linear(hidden_size, hidden_size)
        self.attention_hidden = nn.Linear(hidden_size, hidden_size, bias=False)
        self.attention_score = nn.Linear(hidden_size, 1, bias=False)
        self.input_fusion = nn.Linear(embedding_size + hidden_size, embedding_size)
        self.recurrent = nn.LSTM(
            embedding_size,
            hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=False,
            dropout=0,
        )
        self.output_dropout = nn.Dropout(dropout)
        self.output = nn.Linear(hidden_size, vocab_size)

    def project_image(self, features):
        if features.ndim != 3 or features.shape[1] != self.grid_size**2:
            raise ValueError(
                "Expected spatial features with shape [batch, grid_size**2, feature_dim]."
            )
        if features.shape[2] != self.image_projection.in_features:
            raise ValueError("The cached feature dimension does not match the model.")
        return self.image_projection(features)

    def initial_state(self, visual):
        hidden = visual.new_zeros(self.num_layers, len(visual), self.hidden_size)
        initial = (hidden, torch.zeros_like(hidden))
        # Start from zeros, then process the mean image as a separate visual step.
        image_step = visual.mean(dim=1).unsqueeze(1)
        _, state = self.recurrent(image_step, initial)
        return state

    def image_state(self, features):
        return self.initial_state(self.project_image(features))

    def word_step(self, token_ids, visual, visual_keys, state):
        query = self.attention_hidden(state[0][-1]).unsqueeze(1)
        scores = self.attention_score(torch.tanh(visual_keys + query)).squeeze(-1)
        # All 49 positions remain available, including image-padding regions.
        attention = torch.softmax(scores, dim=1)
        context = torch.sum(visual * attention.unsqueeze(-1), dim=1)

        word = self.embedding_dropout(self.embedding(token_ids))
        recurrent_input = self.input_fusion(torch.cat([word, context], dim=-1))
        output, state = self.recurrent(recurrent_input.unsqueeze(1), state)
        logits = self.output(self.output_dropout(output[:, 0]))
        return logits, state, attention

    def forward_with_attention(self, features, input_ids):
        if input_ids.ndim != 2 or input_ids.shape[1] < 1 or len(features) != len(input_ids):
            raise ValueError("Each image needs a nonempty input token sequence.")
        visual = self.project_image(features)
        visual_keys = self.attention_visual(visual)
        state = self.initial_state(visual)
        logits = []
        attention = []

        # Input: <start> and previous words. Each output predicts the next word.
        # The visual step above has no word target and is excluded from these outputs.
        for token_ids in input_ids.unbind(dim=1):
            next_logits, state, weights = self.word_step(token_ids, visual, visual_keys, state)
            logits.append(next_logits)
            attention.append(weights)
        return torch.stack(logits, dim=1), torch.stack(attention, dim=1)

    def forward(self, features, input_ids):
        logits, _ = self.forward_with_attention(features, input_ids)
        return logits

    @torch.no_grad()
    def greedy(self, features, start_id, end_id, max_steps=33):
        if max_steps < 1:
            raise ValueError("max_steps must be positive.")
        was_training = self.training
        self.eval()
        try:
            visual = self.project_image(features)
            visual_keys = self.attention_visual(visual)
            state = self.initial_state(visual)
            current = torch.full(
                (len(features),), start_id, dtype=torch.long, device=features.device
            )
            finished = torch.zeros(len(features), dtype=torch.bool, device=features.device)
            predictions = [[] for _ in range(len(features))]

            for _ in range(max_steps):
                logits, state, _ = self.word_step(current, visual, visual_keys, state)
                logits[:, self.pad_id] = -torch.inf
                logits[:, start_id] = -torch.inf
                next_ids = logits.argmax(dim=-1)
                for index, (token_id, done) in enumerate(zip(next_ids.tolist(), finished.tolist())):
                    if not done and token_id != end_id:
                        predictions[index].append(token_id)
                finished |= next_ids.eq(end_id)
                if finished.all():
                    break
                # Keep both LSTM states and feed back only the generated token.
                current = next_ids
            return predictions
        finally:
            self.train(was_training)


class CaptionSpatialTransformer(nn.Module):
    def __init__(
        self,
        feature_dim,
        vocab_size,
        pad_id,
        d_model=512,
        num_layers=3,
        nhead=8,
        dim_feedforward=2048,
        dropout=0.3,
        grid_size=7,
        max_positions=34,
    ):
        super().__init__()
        if d_model % nhead != 0:
            raise ValueError("d_model must be divisible by nhead.")
        if num_layers < 1 or grid_size < 1 or max_positions < 1:
            raise ValueError("Use positive layer, grid and position counts.")

        self.decoder_type = "transformer_cross_attention"
        self.d_model = d_model
        self.hidden_size = d_model
        self.num_layers = num_layers
        self.pad_id = pad_id
        self.grid_size = grid_size
        self.max_positions = max_positions

        self.image_projection = nn.Linear(feature_dim, d_model)
        self.row_embedding = nn.Embedding(grid_size, d_model)
        self.column_embedding = nn.Embedding(grid_size, d_model)
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.position_embedding = nn.Embedding(max_positions, d_model)
        self.embedding_dropout = nn.Dropout(dropout)
        # Construct each block independently instead of copying one initialized block.
        self.layers = nn.ModuleList(
            [
                nn.TransformerDecoderLayer(
                    d_model=d_model,
                    nhead=nhead,
                    dim_feedforward=dim_feedforward,
                    dropout=dropout,
                    activation="relu",
                    batch_first=True,
                    norm_first=True,
                )
                for _ in range(num_layers)
            ]
        )
        self.final_norm = nn.LayerNorm(d_model)
        self.output_dropout = nn.Dropout(dropout)
        self.output = nn.Linear(d_model, vocab_size)

        for embedding in [
            self.embedding,
            self.position_embedding,
            self.row_embedding,
            self.column_embedding,
        ]:
            nn.init.normal_(embedding.weight, mean=0.0, std=0.02)
        with torch.no_grad():
            self.embedding.weight[pad_id].zero_()

    def prepare_memory(self, features):
        if features.ndim != 3 or features.shape[1] != self.grid_size**2:
            raise ValueError(
                "Expected spatial features with shape [batch, grid_size**2, feature_dim]."
            )
        if features.shape[2] != self.image_projection.in_features:
            raise ValueError("The cached feature dimension does not match the model.")
        # flatten(2).transpose(1, 2) stores the original CNN grid in row-major order.
        positions = torch.arange(self.grid_size**2, device=features.device)
        rows = positions // self.grid_size
        columns = positions % self.grid_size
        visual_positions = self.row_embedding(rows) + self.column_embedding(columns)
        return self.image_projection(features) + visual_positions.unsqueeze(0)

    def decode_memory(self, memory, input_ids):
        if input_ids.ndim != 2 or input_ids.shape[1] < 1 or len(memory) != len(input_ids):
            raise ValueError("Each image needs a nonempty input token sequence.")
        length = input_ids.shape[1]
        if length > self.max_positions:
            raise ValueError("The input words exceed max_positions.")
        positions = torch.arange(length, device=input_ids.device)
        tokens = self.embedding(input_ids) * self.d_model**0.5
        tokens = self.embedding_dropout(tokens + self.position_embedding(positions).unsqueeze(0))
        causal_mask = torch.triu(
            torch.ones(length, length, dtype=torch.bool, device=input_ids.device), diagonal=1
        )
        padding_mask = input_ids.eq(self.pad_id)
        for layer in self.layers:
            # Only text is causal. Every word can attend to every visual location.
            tokens = layer(tokens, memory, tgt_mask=causal_mask, tgt_key_padding_mask=padding_mask)
        return self.output(self.output_dropout(self.final_norm(tokens)))

    def forward(self, features, input_ids):
        # There is no image token in the text sequence, so every output has a word target.
        return self.decode_memory(self.prepare_memory(features), input_ids)

    @torch.no_grad()
    def greedy(self, features, start_id, end_id, max_steps=33):
        if not 1 <= max_steps <= self.max_positions:
            raise ValueError("max_steps must fit the learned word positions.")
        was_training = self.training
        self.eval()
        try:
            # The image memory is unchanged during generation and is prepared only once.
            memory = self.prepare_memory(features)
            prefix = torch.full(
                (len(features), 1), start_id, dtype=torch.long, device=features.device
            )
            finished = torch.zeros(len(features), dtype=torch.bool, device=features.device)
            predictions = [[] for _ in range(len(features))]
            for _ in range(max_steps):
                logits = self.decode_memory(memory, prefix)[:, -1]
                logits[:, self.pad_id] = -torch.inf
                logits[:, start_id] = -torch.inf
                next_ids = logits.argmax(dim=-1)
                for index, (token_id, done) in enumerate(zip(next_ids.tolist(), finished.tolist())):
                    if not done and token_id != end_id:
                        predictions[index].append(token_id)
                finished |= next_ids.eq(end_id)
                if finished.all():
                    break
                prefix = torch.cat([prefix, next_ids.unsqueeze(1)], dim=1)
            return predictions
        finally:
            self.train(was_training)


In [ ]:
def make_spatial_config(decoder, **changes):
    if decoder not in CONTROL_RUNS:
        raise ValueError("Choose lstm_attention or transformer_cross_attention.")
    config = {
        "encoder": "resnet50",
        "decoder": decoder,
        "variant": "spatial_depth_3",
        "control_run": CONTROL_RUNS[decoder],
        "seed": 42,
        "embedding_size": 512,
        "hidden_size": 512,
        "num_layers": 3,
        "dropout": 0.3,
        "image_layernorm": False,
        "recurrent_dropout": 0.0,
        "grid_size": 7,
        "visual_tokens": 49,
        "feature_dim": 2048,
        "feature_storage": "float16",
        "image_initialization": (
            "mean visual step" if decoder == "lstm_attention" else "cross-attention memory"
        ),
        "attention": "additive" if decoder == "lstm_attention" else "multihead cross-attention",
        "learning_rate": 0.001,
        "batch_size": TRAIN_BATCH_SIZE,
        "max_epochs": 100,
        "patience": 3,
        "gradient_clip": 5.0,
        "amp": True,
        "max_generation_steps": 33,
        "checkpoint_every_batches": 200,
        "print_every_batches": 100,
        "overfit_images": 4,
        "overfit_steps": 2000,
        "overfit_target_loss": 0.1,
        "run_name": f"resnet50_{decoder}_depth3_seed42",
    }
    if decoder == "transformer_cross_attention":
        config.update(
            d_model=512,
            nhead=8,
            dim_feedforward=2048,
            max_positions=34,
            norm_first=True,
            position_encoding="learned text and visual row/column",
        )
    if set(changes) - set(config):
        raise ValueError("Unknown configuration setting.")
    config.update(changes)
    assert config["decoder"] == decoder and config["encoder"] == "resnet50"
    assert config["embedding_size"] == config["hidden_size"] == 512
    assert config["num_layers"] == 3 and config["dropout"] == 0.3
    assert config["grid_size"] == 7 and config["visual_tokens"] == 49
    assert config["control_run"] == CONTROL_RUNS[decoder]
    for key in [
        "batch_size",
        "max_epochs",
        "patience",
        "checkpoint_every_batches",
        "overfit_steps",
    ]:
        assert type(config[key]) is int and config[key] > 0
    return config


def build_model(data, config, device):
    common = {
        "feature_dim": 2048,
        "vocab_size": len(data["id_to_token"]),
        "pad_id": data["ids"]["pad"],
        "num_layers": config["num_layers"],
        "dropout": config["dropout"],
        "grid_size": config["grid_size"],
    }
    if config["decoder"] == "lstm_attention":
        model = CaptionSpatialLSTM(**common, embedding_size=512, hidden_size=512)
    elif config["decoder"] == "transformer_cross_attention":
        model = CaptionSpatialTransformer(
            **common,
            d_model=512,
            nhead=config["nhead"],
            dim_feedforward=config["dim_feedforward"],
            max_positions=config["max_positions"],
        )
    else:
        raise ValueError("Unknown spatial decoder.")
    return model.to(device)


def implementation_for(config):
    return SPATIAL_IMPLEMENTATION_ID


def attach_spatial_features(data):
    # Keep global-vector data intact: only this shallow copy uses the new cache.
    spatial_data = dict(data)
    files = load_spatial_feature_files(data, require_complete=True)
    spatial_data["feature_files"] = {**data["feature_files"], "resnet50": files}
    spatial_data["provenance"] = dict(data["provenance"])
    spatial_data["provenance"]["feature_cache_keys"] = {
        "resnet50": {source: entry["cache_key"] for source, entry in files.items()}
    }
    spatial_data["provenance"]["spatial_layout"] = "49x2048,row-major 7x7,float16 storage"
    return spatial_data


In [ ]:
spatial_data = None
try:
    spatial_data = attach_spatial_features(data)
    print(
        "Spatial caches are ready:",
        {
            source: item["shape"]
            for source, item in spatial_data["feature_files"]["resnet50"].items()
        },
    )
except FileNotFoundError as error:
    print(error)
    print("Enable EXTRACT_SPATIAL_FEATURES and rerun the extraction cell before training.")

if (
    not SPATIAL_DECODERS
    or len(set(SPATIAL_DECODERS)) != len(SPATIAL_DECODERS)
    or not set(SPATIAL_DECODERS) <= set(CONTROL_RUNS)
):
    raise ValueError("Select the spatial decoder names without duplicates.")
configs = [make_spatial_config(decoder) for decoder in SPATIAL_DECODERS]
save_json(
    SPATIAL_EXPERIMENT_DIR / "experiment_plan.json",
    {"implementation": SPATIAL_IMPLEMENTATION_ID, "configs": configs, "controls": CONTROL_RUNS},
)
for config in configs:
    print("New run:", config["run_name"], "| saved control:", config["control_run"])


## 6. Metrics and preflight checks

Use the same pycocoevalcap 1.2 CIDEr-D and corpus BLEU-1/4 as the controls, including the project's identical regex tokenization for predictions and original references. BLEU uses 0–1; CIDEr-D uses its native scale (including the scorer's factor 10). These scores are not official COCO leaderboard PTB results.

Check shifted captions, gradients, image conditioning, attention/causal masks and greedy generation. A temporary model with dropout disabled must memorize four fixed training images; it gets up to 2,000 steps and is discarded before the actual run.

In [ ]:
class MetricEvaluator:
    def __init__(self, settings):
        version = package_metadata.version("pycocoevalcap")
        if version != "1.2":
            raise RuntimeError("Use pycocoevalcap==1.2 so all experiments use the same scorers.")
        self.token_pattern = settings["token_pattern"]
        re.compile(self.token_pattern)
        self.metadata = {
            "package": "pycocoevalcap",
            "version": version,
            "bleu": {
                "implementation": "Bleu(4)",
                "aggregation": "corpus",
                "scale": "0..1; no multiplication by 100",
            },
            "cider_d": {
                "implementation": "Cider(n=4,sigma=6.0)",
                "ngrams": 4,
                "sigma": 6.0,
                "scale": "native score, including the implementation's factor 10; no extra multiplication",
                "document_frequency": "all reference sets in the evaluated split",
            },
            "tokenization": {
                "name": "project_regex_v1",
                "normalization": "NFKC,lowercase,smart quotes,whitespace",
                "token_pattern": self.token_pattern,
                "reference_vocabulary_mapping": False,
                "note": "Same tokenizer for predictions and raw references; not COCO leaderboard PTB tokenization.",
            },
            "source_sha256": {
                name: hashlib.sha256(Path(inspect.getfile(cls)).read_bytes()).hexdigest()
                for name, cls in [
                    ("bleu", Bleu),
                    ("bleu_scorer", BleuScorer),
                    ("cider", Cider),
                    ("cider_scorer", CiderScorer),
                ]
            },
        }

    def tokenize(self, text):
        text = unicodedata.normalize("NFKC", text).lower()
        text = text.translate(
            str.maketrans(
                {
                    "\u2018": "'",
                    "\u2019": "'",
                    "\u02bc": "'",
                    "\u201c": '"',
                    "\u201d": '"',
                    "\u200b": " ",
                    "\ufeff": " ",
                }
            )
        )
        text = " ".join(text.split())
        return " ".join(re.findall(self.token_pattern, text))

    def score(self, records, expected_ids=None):
        # One corpus call is essential: averaging batch BLEU/CIDEr is incorrect.
        if len(records) < 2:
            raise ValueError("CIDEr-D needs a corpus with at least two images.")
        gts = {}
        res = {}
        for item in records:
            image_id = int(item["image_id"])
            if image_id in gts:
                raise ValueError(f"Duplicate prediction for image {image_id}")
            references = item["references"]
            if not isinstance(references, list) or not references:
                raise ValueError(f"Missing human references for image {image_id}")
            if not all(isinstance(text, str) and text.strip() for text in references):
                raise ValueError(f"Empty or invalid human reference for image {image_id}")
            if not isinstance(item["prediction"], str):
                raise TypeError("Each prediction must be a string.")
            gts[image_id] = [self.tokenize(text) for text in references]
            if not all(gts[image_id]):
                raise ValueError(f"A reference has no tokens for image {image_id}")
            res[image_id] = [self.tokenize(item["prediction"])]
        if expected_ids is not None:
            expected = [int(image_id) for image_id in expected_ids]
            if len(expected) != len(set(expected)) or set(gts) != set(expected):
                raise ValueError("Predictions do not cover exactly the requested evaluation split.")
        bleu, _ = Bleu(4).compute_score(gts, res, verbose=0)
        cider_d, _ = Cider(n=4, sigma=6.0).compute_score(gts, res)
        scores = {"bleu1": float(bleu[0]), "bleu4": float(bleu[3]), "cider_d": float(cider_d)}
        if not all(np.isfinite(value) for value in scores.values()):
            raise ValueError("The metric implementation returned a non-finite score.")
        return scores


def check_metrics(evaluator):
    # Distinct documents keep CIDEr's corpus IDF meaningful in this small check.
    sentences = [
        "A red bus drives down the road.",
        "Two cats sleep on the green sofa.",
        "A woman holds a yellow umbrella.",
    ]
    records = [
        {"image_id": index, "prediction": text, "references": [text, text]}
        for index, text in enumerate(sentences)
    ]
    exact = evaluator.score(records, expected_ids=range(len(records)))
    assert abs(exact["bleu1"] - 1) < 1e-6 and abs(exact["bleu4"] - 1) < 1e-6
    assert abs(exact["cider_d"] - 10) < 1e-6
    unrelated = [
        {**row, "prediction": "unrelated vocabulary never matches anything"} for row in records
    ]
    poor = evaluator.score(unrelated)
    assert poor["bleu1"] < 1e-6 and poor["bleu4"] < 1e-6 and poor["cider_d"] == 0
    empty = evaluator.score([{**row, "prediction": ""} for row in records])
    assert empty["cider_d"] == 0 and empty["bleu1"] < 1e-6
    # These are library corpus scores, not sentence scores averaged by our code.
    gts = {
        row["image_id"]: [evaluator.tokenize(text) for text in row["references"]] for row in records
    }
    res = {row["image_id"]: [evaluator.tokenize(row["prediction"])] for row in records}
    direct_bleu, _ = Bleu(4).compute_score(gts, res, verbose=0)
    direct_cider, _ = Cider(n=4, sigma=6.0).compute_score(gts, res)
    assert exact == {
        "bleu1": float(direct_bleu[0]),
        "bleu4": float(direct_bleu[3]),
        "cider_d": float(direct_cider),
    }
    try:
        evaluator.score(records, expected_ids=[0, 1, 99])
    except ValueError:
        pass
    else:
        raise AssertionError("Incomplete evaluation corpus was accepted.")
    print(
        "Metric checks passed: exact captions, wrong captions, empty predictions and full-corpus coverage."
    )
    return {"exact": exact, "unrelated": poor, "empty_predictions": empty}


In [ ]:
def tokens_to_text(token_ids, id_to_token):
    skipped = {"<pad>", "<start>", "<end>"}
    return " ".join(
        id_to_token[int(index)] for index in token_ids if id_to_token[int(index)] not in skipped
    )


def check_caption_shift(caption_ids, ids):
    # Example: <start> a cat <end> -> inputs <start> a cat; targets a cat <end>.
    values = [int(value) for value in caption_ids if int(value) != ids["pad"]]
    assert len(values) >= 2 and values[0] == ids["start"] and values[-1] == ids["end"]

    inputs = values[:-1]
    targets = values[1:]
    assert len(inputs) == len(targets) == len(values) - 1
    assert inputs[0] == ids["start"] and targets[-1] == ids["end"]

    return {"caption": values, "input": inputs, "target": targets, "text_length": len(values) - 1}


def overfit_small_set(batch, data, config, device):
    # Keep the experiment's architecture; disable dropout only for this wiring check.
    test_config = dict(config)
    test_config["dropout"] = 0.0
    model = build_model(data, test_config, device)
    ids = data["ids"]
    vocab_size = len(data["id_to_token"])
    feature_dim = FEATURE_DIMS[config["encoder"]]
    max_steps = config["overfit_steps"]
    learning_rate = config["learning_rate"]
    target_loss = config["overfit_target_loss"]
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.CrossEntropyLoss(ignore_index=ids["pad"], reduction="sum")

    features = batch["features"].to(device)
    captions = batch["captions"].to(device)

    inputs = captions[:, :-1]
    targets = captions[:, 1:]
    token_count = int(targets.ne(ids["pad"]).sum())
    assert token_count > 0 and max_steps > 0

    references = []
    for row in captions.tolist():
        references.append(
            [token for token in row if token not in {ids["pad"], ids["start"], ids["end"]}]
        )

    start = time.perf_counter()
    initial_loss = None
    checks = []
    passed = False

    for step in range(1, max_steps + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)

        logits = model(features, inputs)
        loss = criterion(logits.reshape(-1, vocab_size), targets.reshape(-1)) / token_count
        assert torch.isfinite(loss), "Non-finite small-set loss."
        if initial_loss is None:
            initial_loss = float(loss.detach())

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()

        if step % 25 == 0 or step == max_steps:
            model.eval()
            with torch.no_grad():
                logits = model(features, inputs)
                checked_loss = float(
                    criterion(logits.reshape(-1, vocab_size), targets.reshape(-1)) / token_count
                )
                mask = targets.ne(ids["pad"])
                token_accuracy = float(logits.argmax(-1).eq(targets)[mask].float().mean())

            generated = model.greedy(
                features, ids["start"], ids["end"], max_steps=captions.shape[1] - 1
            )
            exact = sum(
                prediction == reference for prediction, reference in zip(generated, references)
            ) / len(references)

            checks.append(
                {
                    "step": step,
                    "loss": checked_loss,
                    "teacher_forced_token_accuracy": token_accuracy,
                    "greedy_exact_match": exact,
                }
            )
            print(
                f"{config['decoder']} / {config['variant']} small-set step {step}: "
                f"loss={checked_loss:.4f}, token accuracy={token_accuracy:.1%}, greedy exact={exact:.1%}",
                flush=True,
            )
            if checked_loss <= target_loss and exact == 1.0:
                passed = True
                break

    if device.type == "cuda":
        torch.cuda.synchronize(device)

    result = {
        "decoder": config["decoder"],
        "variant": config["variant"],
        "hidden_size": config["hidden_size"],
        "num_layers": config["num_layers"],
        "image_layernorm": config["image_layernorm"],
        "feature_dim": feature_dim,
        "images": len(features),
        "dropout": 0,
        "learning_rate": learning_rate,
        "steps": step,
        "initial_loss": initial_loss,
        "final_loss": checks[-1]["loss"],
        "greedy_exact_match": checks[-1]["greedy_exact_match"],
        "teacher_forced_token_accuracy": checks[-1]["teacher_forced_token_accuracy"],
        "elapsed_seconds": time.perf_counter() - start,
        "passed": passed,
        "checks": checks,
        "note": "This temporary model is discarded. Re-seed and create a new model for the real experiment.",
    }
    del model, optimizer, logits, loss, features, captions
    return result


@torch.no_grad()
def check_spatial_attention(model, features, ids):
    was_training = model.training
    model.eval()
    try:
        features = features[:2]
        words = [i for i in range(model.embedding.num_embeddings) if i not in ids.values()][:3]
        assert len(words) == 3
        probe = torch.tensor([ids["start"], *words], device=features.device).repeat(
            len(features), 1
        )
        original = model(features, probe)
        tolerance = 0.005 if original.dtype in [torch.float16, torch.bfloat16] else 1e-4

        future_changed = probe.clone()
        future_changed[:, 2:] = probe[:, 2:].flip(1)
        changed = model(features, future_changed)
        assert torch.allclose(
            original[:, :2], changed[:, :2], atol=tolerance, rtol=tolerance
        ), "Future words changed an earlier prediction."
        padded = torch.cat(
            [probe, torch.full((len(probe), 2), ids["pad"], device=probe.device)], dim=1
        )
        assert torch.allclose(
            original, model(features, padded)[:, :4], atol=tolerance, rtol=tolerance
        ), "Right padding changed valid outputs."

        other_features = torch.zeros_like(features)
        if torch.equal(other_features, features):
            other_features = torch.ones_like(features)
        assert (
            original - model(other_features, probe)
        ).abs().max() > 1e-6, "Predictions do not depend on the image."
        last_changed = features.clone()
        last_changed[:, -1] += torch.linspace(-3, 3, features.shape[-1], device=features.device)
        assert (
            original[:, 0] - model(last_changed, probe)[:, 0]
        ).abs().max() > 1e-6, "The first word cannot use the last visual region."

        checks = {
            "causal": True,
            "right_padding": True,
            "image_conditioning": True,
            "last_region_visible": True,
        }
        if model.decoder_type == "lstm_attention":
            state = model.image_state(features)
            expected = (model.num_layers, len(features), model.hidden_size)
            assert state[0].shape == state[1].shape == expected
            logits, weights = model.forward_with_attention(features, probe)
            assert torch.allclose(original, logits, atol=tolerance, rtol=tolerance)
            assert weights.shape == (len(features), probe.shape[1], features.shape[1])
            assert torch.isfinite(weights).all() and (weights >= 0).all()
            assert torch.allclose(
                weights.sum(-1), torch.ones_like(weights.sum(-1)), atol=tolerance, rtol=tolerance
            )
            checks.update(lstm_state_shapes=True, attention_probabilities=True)
        else:
            assert model.decoder_type == "transformer_cross_attention"
            memory = model.prepare_memory(features)
            assert memory.shape == (len(features), features.shape[1], model.hidden_size)
            assert len(model.layers) == model.num_layers
            for name in ["self_attn", "multihead_attn"]:
                weights = [getattr(layer, name).in_proj_weight for layer in model.layers]
                assert len({weight.data_ptr() for weight in weights}) == len(weights)
                assert all(
                    not torch.equal(weights[i], weights[j])
                    for i in range(len(weights))
                    for j in range(i)
                )
            positions = model.prepare_memory(torch.zeros_like(features))
            assert (positions[:, 1] - positions[:, 0]).abs().max() > 1e-6
            assert (positions[:, model.grid_size] - positions[:, 0]).abs().max() > 1e-6

            # An internal PAD isolates its key mask from the causal mask.
            internal_pad = probe.clone()
            internal_pad[:, 1] = ids["pad"]
            before = model(features, internal_pad)[:, [0, 2, 3]]
            saved_pad = model.embedding.weight[ids["pad"]].clone()
            try:
                model.embedding.weight[ids["pad"]].copy_(
                    torch.linspace(-4, 4, model.hidden_size, device=features.device)
                )
                after = model(features, internal_pad)[:, [0, 2, 3]]
                assert torch.allclose(
                    before, after, atol=tolerance, rtol=tolerance
                ), "A masked PAD token changed valid outputs."
            finally:
                model.embedding.weight[ids["pad"]].copy_(saved_pad)
            checks.update(
                memory_shape=True,
                independent_layers=True,
                row_column_positions=True,
                internal_padding=True,
            )
        return {**checks, "passed": True}
    finally:
        model.train(was_training)


def smoke_model_check(batch, data, config, device):
    model = build_model(data, config, device)
    ids = data["ids"]
    vocab_size = len(data["id_to_token"])
    features = batch["features"].to(device).detach().clone().requires_grad_(True)
    captions = batch["captions"].to(device)
    feature_dim = FEATURE_DIMS[config["encoder"]]
    assert features.shape == (len(captions), 49, feature_dim) and feature_dim == 2048
    inputs, targets = captions[:, :-1], captions[:, 1:]
    token_count = int(targets.ne(ids["pad"]).sum())
    assert token_count > 0
    attention_checks = check_spatial_attention(model, features, ids)

    logits = model(features, inputs)
    assert logits.shape == (*targets.shape, vocab_size) and torch.isfinite(logits).all()
    criterion = nn.CrossEntropyLoss(ignore_index=ids["pad"], reduction="sum")
    loss = criterion(logits.reshape(-1, vocab_size), targets.reshape(-1)) / token_count
    assert torch.isfinite(loss)
    loss.backward()
    for name, parameter in model.named_parameters():
        assert parameter.grad is not None, f"Missing gradient: {name}"
        assert torch.isfinite(parameter.grad).all(), f"Invalid gradient: {name}"
    assert model.embedding.weight.grad[ids["pad"]].eq(0).all()
    assert model.image_projection.weight.grad.abs().sum() > 0
    assert features.grad is not None and torch.isfinite(features.grad).all()
    assert (features.grad.abs().sum(-1) > 0).all(), "A visual region received no input gradient."
    grad_norm = nn.utils.clip_grad_norm_(model.parameters(), 5.0)
    assert torch.isfinite(grad_norm)
    generated = model.greedy(features[:2], ids["start"], ids["end"], max_steps=8)
    assert len(generated) == min(2, len(features))
    assert all(
        len(row) <= 8 and not set(row) & {ids["pad"], ids["start"], ids["end"]} for row in generated
    )

    # This temporary model is discarded, so force predictable logits to test stopping.
    with torch.no_grad():
        model.output.weight.zero_()
        model.output.bias.fill_(-10)
        model.output.bias[ids["pad"]] = 100
        model.output.bias[ids["start"]] = 90
        model.output.bias[ids["end"]] = 80
        calls = []
        hook = model.output.register_forward_hook(lambda module, args, output: calls.append(1))
        try:
            stopped = model.greedy(features[:2], ids["start"], ids["end"], max_steps=8)
            assert stopped == [[] for _ in range(min(2, len(features)))] and len(calls) == 1
            word_id = next(i for i in range(vocab_size) if i not in ids.values())
            model.output.bias[ids["end"]] = -10
            model.output.bias[word_id] = 80
            calls.clear()
            limited = model.greedy(features[:2], ids["start"], ids["end"], max_steps=3)
            assert limited == [[word_id] * 3 for _ in range(min(2, len(features)))]
            assert len(calls) == 3
        finally:
            hook.remove()
    attention_checks.update(
        all_regions_have_gradients=True,
        greedy_special_tokens=True,
        greedy_end_stop=True,
        greedy_length_limit=True,
    )
    result = {
        "decoder": config["decoder"],
        "variant": config["variant"],
        "hidden_size": config["hidden_size"],
        "num_layers": config["num_layers"],
        "feature_dim": feature_dim,
        "visual_regions": 49,
        "batch_size": len(features),
        "loss": float(loss.detach()),
        "target_tokens": token_count,
        "gradient_norm_before_clip": float(grad_norm),
        "trainable_parameters": sum(p.numel() for p in model.parameters() if p.requires_grad),
        "output_shape": list(logits.shape),
        "generation_lengths": [len(row) for row in generated],
        "attention_checks": attention_checks,
        "passed": True,
    }
    del model, logits, loss, features, captions
    return result


In [ ]:
def run_preflight(data, config, device, check_dir):
    check_dir = Path(check_dir)
    identity = {
        "implementation": implementation_for(config),
        "config": config,
        "data": data["provenance"],
        "device": device_details(device),
    }
    key = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()
    path = (
        check_dir / f"{config['encoder']}_{config['decoder']}_{config['variant']}_{key[:12]}.json"
    )
    if path.exists():
        saved = json.loads(path.read_text(encoding="utf-8"))
        if saved["identity"] == identity and saved["passed"]:
            print("Loaded successful batch/backward/greedy and small-set checks.")
            return saved
    seed_everything(config["seed"])
    dataset = CachedCaptionDataset(data, config["encoder"], "train", config["seed"])
    try:
        batch = next(iter(make_loader(dataset, config["batch_size"], seed=config["seed"])))
        smoke = smoke_model_check(batch, data, config, device)
        # A short fixed caption per image makes this a quick wiring check, not an experiment.
        chosen = [
            index
            for index, image_id in enumerate(dataset.image_ids)
            if len(data["tokens"]["train"][image_id][0]) <= 12
        ][: config["overfit_images"]]
        assert len(chosen) == config["overfit_images"]
        dataset.caption_choices = [0] * len(dataset)
        tiny = next(iter(make_loader(dataset, len(chosen), chosen, seed=config["seed"])))
        seed_everything(config["seed"])
        overfit = overfit_small_set(tiny, data, config, device)
        result = {
            "identity": identity,
            "smoke": smoke,
            "overfit": overfit,
            "image_ids": tiny["image_ids"],
            "passed": bool(smoke["passed"] and overfit["passed"]),
        }
        save_json(path, result)
        if not result["passed"]:
            raise RuntimeError(
                "Small-set memorization did not pass. Inspect its saved report before full training."
            )
        return result
    finally:
        dataset.close()


@torch.no_grad()
def evaluate_model(model, dataset, data, config, device, evaluator):
    if dataset.split not in ["val", "test"]:
        raise ValueError("Evaluation uses validation or the labeled test.")
    model.eval()

    criterion = nn.CrossEntropyLoss(ignore_index=data["ids"]["pad"], reduction="sum")
    total_loss = 0.0
    total_tokens = 0
    records = []
    amp_enabled = config["amp"] and device.type == "cuda"
    for batch in make_loader(dataset, config["batch_size"], seed=config["seed"]):
        features = batch["features"].to(device, non_blocking=True)
        captions = batch["captions"].to(device, non_blocking=True)
        targets = captions[:, 1:]

        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=amp_enabled):
            logits = model(features, captions[:, :-1])
            loss_sum = criterion(logits.reshape(-1, len(data["id_to_token"])), targets.reshape(-1))
            predicted = model.greedy(
                features, data["ids"]["start"], data["ids"]["end"], config["max_generation_steps"]
            )

        total_loss += float(loss_sum)
        total_tokens += int(targets.ne(data["ids"]["pad"]).sum())
        for image_id, token_ids, references in zip(
            batch["image_ids"], predicted, batch["references"]
        ):
            records.append(
                {
                    "image_id": image_id,
                    "prediction": tokens_to_text(token_ids, data["id_to_token"]),
                    "token_ids": token_ids,
                    "references": references,
                }
            )
    assert total_tokens > 0 and len(records) == len(dataset)
    scores = evaluator.score(records, expected_ids=data["split_ids"][dataset.split])
    return {"loss": total_loss / total_tokens, **scores}, records


In [ ]:
evaluator = MetricEvaluator(data["text_settings"])
for control in CONTROL_RUNS.values():
    saved = json.loads(
        (GLOBAL_EXPERIMENT_DIR / control / "config.json").read_text(encoding="utf-8")
    )
    assert saved["metrics"] == evaluator.metadata
    for key in ["source_id", "split_id", "vocab_sha256", "prepared_sha256", "annotations_sha256"]:
        assert saved["data"][key] == data["provenance"][key]
if spatial_data is not None:
    shift_dataset = CachedCaptionDataset(spatial_data, "resnet50", "train", 42)
    try:
        shift_example = check_caption_shift(shift_dataset[0]["caption"], data["ids"])
        print(
            "Teacher-forcing input:",
            " ".join(data["id_to_token"][i] for i in shift_example["input"]),
        )
        print(
            "Next-token target:", " ".join(data["id_to_token"][i] for i in shift_example["target"])
        )
    finally:
        shift_dataset.close()
if RUN_PREFLIGHT:
    if spatial_data is None:
        raise RuntimeError("Complete the spatial cache first.")
    metric_checks = check_metrics(evaluator)
    preflight_results = [
        run_preflight(spatial_data, config, device, SPATIAL_EXPERIMENT_DIR / "checks")
        for config in configs
    ]
else:
    print("Explicit preflight is off. Each new training run still requires it.")


## 7. Train, time, checkpoint and resume

The training/evaluation loop keeps the old protocol: next-token teacher forcing, token-weighted CrossEntropy ignoring PAD, no softmax before loss, AMP unscale before clipping the total gradient norm to 5.0. Validation generates independently and scores the whole 5,000-image corpus.

Save best/last full checkpoints and plain weights, optimizer/scaler/RNG states, early-stopping progress, vocabulary, configurations, histories and raw-reference predictions. Interrupted work resumes from the last committed step. Active timers exclude pauses and preflight. The new group is `experiments/resnet50_spatial_v1`; the 79 control runs are read-only.

In [ ]:
def write_checkpoint(path, model, optimizer, scaler, state, identity):
    # Call only between complete optimizer steps. An interrupted temporary write is ignored.
    checkpoint = {
        "format_version": 1,
        "identity": identity,
        "state": state,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scaler": scaler.state_dict(),
        "rng": rng_state(),
        "order_seed": identity["config"]["seed"] + state["epoch"],
        "caption_seed": identity["config"]["seed"] + state["epoch"],
    }
    path = Path(path)
    temporary = path.with_suffix(".pt.tmp")
    torch.save(checkpoint, temporary)
    temporary.replace(path)
    save_weights(path.with_name(path.stem + "_weights.pt"), model)


def load_checkpoint(path, model, optimizer, scaler, identity):
    # Load only this project's own checkpoints: they contain Python/NumPy RNG states.
    checkpoint = torch.load(path, map_location="cpu", weights_only=False)
    assert checkpoint["format_version"] == 1
    if checkpoint["identity"] != identity:
        raise ValueError(
            "Checkpoint settings/data/metric version differ. Restore the same settings or choose a new run_name."
        )
    model.load_state_dict(checkpoint["model"])
    optimizer.load_state_dict(checkpoint["optimizer"])
    if checkpoint["scaler"]:
        scaler.load_state_dict(checkpoint["scaler"])
    elif scaler.is_enabled():
        print(
            "The previous run used no AMP scaler; starting a fresh CUDA scaler after the device change."
        )
    restore_rng(checkpoint["rng"])
    return checkpoint["state"]


def save_experiment_summary(folder, config, state, parameter_count):
    if state["best_record"] is None:
        return
    row = state["best_record"]
    summary = {
        "encoder": config["encoder"],
        "decoder": config["decoder"],
        **{
            key: config[key]
            for key in [
                "variant",
                "embedding_size",
                "hidden_size",
                "num_layers",
                "dropout",
                "image_layernorm",
                "seed",
            ]
        },
        **{
            key: config.get(key)
            for key in [
                "d_model",
                "nhead",
                "dim_feedforward",
                "max_positions",
                "norm_first",
                "position_encoding",
            ]
        },
        "feature_dim": FEATURE_DIMS[config["encoder"]],
        "trainable_parameters": parameter_count,
        "best_epoch": state["best_epoch"],
        "val_loss": row["val_loss"],
        "bleu1": row["bleu1"],
        "bleu4": row["bleu4"],
        "cider_d": row["cider_d"],
        "training_seconds": state["training_seconds"],
        "training_time": format_time(state["training_seconds"]),
        "devices": state["devices"],
        "status": state["phase"],
    }
    save_json(folder / "summary.json", summary)


In [ ]:
def run_experiment(
    data, config, evaluator, device, output_root, resume=True, _after_checkpoint=None
):
    output_root = Path(output_root)
    name = (
        config["run_name"]
        or f"{config['encoder']}_{config['decoder']}_{config['variant']}_seed{config['seed']}"
    )
    if Path(name).name != name or name in [".", ".."]:
        raise ValueError("run_name must be a single folder name.")
    folder = output_root / name
    folder.mkdir(parents=True, exist_ok=True)
    identity = experiment_identity(data, config, evaluator)
    if (folder / "config.json").exists():
        if json.loads((folder / "config.json").read_text(encoding="utf-8")) != identity:
            raise ValueError("Existing experiment has different settings. Use a new run_name.")
        if (folder / "last.pt").exists() and not resume:
            raise ValueError(
                "This experiment already exists. Set resume=True or use a new run_name."
            )
    save_json(folder / "config.json", identity)
    save_json(
        folder / "vocabulary.json",
        {
            "id_to_token": data["id_to_token"],
            "token_to_id": data["token_to_id"],
            "source_sha256": data["provenance"]["vocab_sha256"],
        },
    )
    run_preflight(data, config, device, output_root / "checks")
    # Never train the small-set model. Reset RNG and construct a fresh main model.
    seed_everything(config["seed"])
    model = build_model(data, config, device)
    optimizer = torch.optim.Adam(model.parameters(), lr=config["learning_rate"])
    amp_enabled = config["amp"] and device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=amp_enabled)
    criterion = nn.CrossEntropyLoss(ignore_index=data["ids"]["pad"], reduction="sum")
    parameter_count = sum(
        parameter.numel() for parameter in model.parameters() if parameter.requires_grad
    )
    # Reserve room for full best/last checkpoints, plain weights, and temporary writes.
    required_bytes = parameter_count * 4 * 10 + 128 * 1024**2
    if shutil.disk_usage(folder).free < required_bytes:
        raise OSError(
            f"Not enough free space for this run. Free at least {required_bytes / 1024**3:.2f} GiB or move RESULTS_ROOT."
        )
    details = device_details(device)
    state = {
        "epoch": 1,
        "next_index": 0,
        "phase": "train",
        "train_loss_sum": 0.0,
        "train_token_count": 0,
        "best_score": -float("inf"),
        "best_epoch": None,
        "best_record": None,
        "bad_epochs": 0,
        "history": [],
        "training_seconds": 0.0,
        "epoch_seconds": 0.0,
        "devices": [details],
    }
    if (folder / "last.pt").exists():
        state = load_checkpoint(folder / "last.pt", model, optimizer, scaler, identity)
        if details not in state["devices"]:
            state["devices"].append(details)
            print(
                "Device changed on resume. Timing combines devices; do not compare it as an architecture-only effect."
            )
        print(
            f"Resuming epoch {state['epoch']}, next image index {state['next_index']}, phase {state['phase']}."
        )
    else:
        write_checkpoint(folder / "last.pt", model, optimizer, scaler, state, identity)
    if state["phase"] == "finished":
        # The checkpoint is authoritative if interruption happened between its write and the JSON files.
        save_json(folder / "history.json", state["history"])
        save_experiment_summary(folder, config, state, parameter_count)
        save_json(folder / "status.json", {"status": "finished", "epoch": state["epoch"]})
        save_weights(folder / "last_weights.pt", model)
        print(
            f"Experiment is already complete. Total active time: {format_time(state['training_seconds'])}"
        )
        return json.loads((folder / "summary.json").read_text(encoding="utf-8"))
    train_data = CachedCaptionDataset(data, config["encoder"], "train", config["seed"])
    val_data = CachedCaptionDataset(data, config["encoder"], "val", config["seed"])
    previous_seconds = state["training_seconds"]
    run_started = time.perf_counter()
    epoch_started = run_started
    previous_epoch_seconds = state["epoch_seconds"]

    def commit():
        if device.type == "cuda":
            torch.cuda.synchronize(device)
        state["training_seconds"] = previous_seconds + time.perf_counter() - run_started
        state["epoch_seconds"] = previous_epoch_seconds + time.perf_counter() - epoch_started
        if state["phase"] in ["epoch_done", "finished"] and state["history"]:
            state["history"][-1]["epoch_seconds"] = state["epoch_seconds"]
        write_checkpoint(folder / "last.pt", model, optimizer, scaler, state, identity)
        save_json(folder / "history.json", state["history"])
        save_experiment_summary(folder, config, state, parameter_count)

    try:
        while state["epoch"] <= config["max_epochs"]:
            epoch = state["epoch"]
            if state["phase"] == "epoch_done":
                state.update(
                    epoch=epoch + 1,
                    next_index=0,
                    phase="train",
                    train_loss_sum=0.0,
                    train_token_count=0,
                    epoch_seconds=0.0,
                )
                continue

            epoch_started = time.perf_counter()
            previous_epoch_seconds = state["epoch_seconds"]
            starting_index = state["next_index"]
            print(
                f"Epoch {epoch}/{config['max_epochs']} | {config['encoder']} + {config['decoder']} + {config['variant']} | active total {format_time(state['training_seconds'])}",
                flush=True,
            )

            train_data.set_epoch(epoch)
            order = training_order(len(train_data), config["seed"], epoch)
            model.train()

            for batch in make_loader(
                train_data,
                config["batch_size"],
                order[state["next_index"] :],
                config["seed"] + epoch,
            ):
                # 1. Move a batch to the device and shift captions by one token.
                features = batch["features"].to(device, non_blocking=True)
                captions = batch["captions"].to(device, non_blocking=True)
                targets = captions[:, 1:]
                count = int(targets.ne(data["ids"]["pad"]).sum())

                # 2. Teacher forcing: predict the next word from previous words.
                optimizer.zero_grad(set_to_none=True)
                with torch.autocast(
                    device_type=device.type, dtype=torch.float16, enabled=amp_enabled
                ):
                    logits = model(features, captions[:, :-1])
                    loss_sum = criterion(
                        logits.reshape(-1, len(data["id_to_token"])), targets.reshape(-1)
                    )
                    loss = loss_sum / count
                if not torch.isfinite(loss):
                    raise FloatingPointError(
                        "Non-finite training loss. Resume the last committed checkpoint after diagnosing it."
                    )

                # 3. Backpropagation, unscale, clipping and optimizer update.
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), config["gradient_clip"])
                scaler.step(optimizer)
                scaler.update()

                # 4. Accumulate loss by valid tokens and record completed images.
                state["train_loss_sum"] += float(loss_sum.detach())
                state["train_token_count"] += count
                state["next_index"] += len(batch["image_ids"])
                batch_number = (state["next_index"] + config["batch_size"] - 1) // config[
                    "batch_size"
                ]
                if batch_number % config["print_every_batches"] == 0:
                    segment_seconds = time.perf_counter() - epoch_started
                    images_done = state["next_index"] - starting_index
                    eta_seconds = (
                        segment_seconds
                        * (len(train_data) - state["next_index"])
                        / max(images_done, 1)
                    )
                    epoch_elapsed = previous_epoch_seconds + segment_seconds
                    total_elapsed = previous_seconds + time.perf_counter() - run_started
                    train_loss = state["train_loss_sum"] / state["train_token_count"]
                    print(
                        f"  {state['next_index']:,}/{len(train_data):,} images | loss {train_loss:.4f} | epoch {format_time(epoch_elapsed)} | total {format_time(total_elapsed)} | train ETA {format_time(eta_seconds)}",
                        flush=True,
                    )

                if batch_number % config["checkpoint_every_batches"] == 0:
                    commit()
                    if (
                        _after_checkpoint is not None
                    ):  # Used only by the interruption/resume verification script.
                        _after_checkpoint(state)
            assert state["next_index"] == len(train_data)
            state["phase"] = "evaluate"
            commit()

            print(
                f"  Training pass finished in {format_time(state['epoch_seconds'])}. Validating all {len(val_data):,} images...",
                flush=True,
            )
            validation_started = time.perf_counter()
            scores, predictions = evaluate_model(model, val_data, data, config, device, evaluator)
            validation_seconds = time.perf_counter() - validation_started
            epoch_seconds = previous_epoch_seconds + time.perf_counter() - epoch_started
            row = {
                "epoch": epoch,
                "train_loss": state["train_loss_sum"] / state["train_token_count"],
                "val_loss": scores["loss"],
                "bleu1": scores["bleu1"],
                "bleu4": scores["bleu4"],
                "cider_d": scores["cider_d"],
                "epoch_seconds": epoch_seconds,
                "validation_seconds": validation_seconds,
            }
            state["history"].append(row)
            prediction_path = folder / "predictions" / f"val_epoch_{epoch:02d}.json"
            save_json(prediction_path, predictions)
            improved = scores["cider_d"] > state["best_score"]
            if improved:
                state.update(
                    best_score=scores["cider_d"], best_epoch=epoch, best_record=row, bad_epochs=0
                )
            else:
                state["bad_epochs"] += 1
            done = state["bad_epochs"] >= config["patience"] or epoch >= config["max_epochs"]
            state["phase"] = "finished" if done else "epoch_done"
            if device.type == "cuda":
                torch.cuda.synchronize(device)
            state["training_seconds"] = previous_seconds + time.perf_counter() - run_started
            state["epoch_seconds"] = previous_epoch_seconds + time.perf_counter() - epoch_started
            if improved:
                write_checkpoint(folder / "best.pt", model, optimizer, scaler, state, identity)
                save_json(folder / "predictions" / "val_best.json", predictions)
            commit()
            print(
                f"Epoch {epoch} done | train {row['train_loss']:.4f} | val {row['val_loss']:.4f} | BLEU-1 {row['bleu1']:.4f} | BLEU-4 {row['bleu4']:.4f} | CIDEr-D {row['cider_d']:.4f} | patience {state['bad_epochs']}/{config['patience']}",
                flush=True,
            )
            print(
                f"  Epoch time {format_time(state['epoch_seconds'])} | validation {format_time(validation_seconds)} | total active time {format_time(state['training_seconds'])}",
                flush=True,
            )
            print(f"  Saved: {folder/'last.pt'} and last_weights.pt", flush=True)
            if improved:
                print("  New best model saved: best.pt and best_weights.pt", flush=True)
            if done:
                break
        save_json(folder / "status.json", {"status": "finished", "epoch": state["epoch"]})
        print(
            f"Finished {config['encoder']} + {config['decoder']} + {config['variant']} in {format_time(state['training_seconds'])}. Best epoch: {state['best_epoch']}",
            flush=True,
        )
        return json.loads((folder / "summary.json").read_text(encoding="utf-8"))
    except BaseException:
        # An interruption may happen inside optimizer.step. Never save that unsafe partial state.
        # last.pt always contains a completed step; resume may repeat work after that checkpoint.
        save_json(
            folder / "status.json",
            {
                "status": "interrupted",
                "resume_from": "last.pt",
                "note": "Resume the latest committed checkpoint. Later uncommitted batches may run again.",
            },
        )
        print(
            f"Stopped. Resume with the same config and resume=True. Checkpoint: {folder/'last.pt'}",
            flush=True,
        )
        raise
    finally:
        train_data.close()
        val_data.close()


In [ ]:
if RUN_SPATIAL_TRAINING:
    if spatial_data is None:
        raise RuntimeError("Complete the spatial feature cache before training.")
    session_started = time.perf_counter()
    try:
        spatial_results = []
        for config in configs:
            spatial_results.append(
                run_experiment(
                    spatial_data, config, evaluator, device, SPATIAL_EXPERIMENT_DIR, resume=True
                )
            )
    finally:
        if device.type == "cuda":
            torch.cuda.synchronize(device)
        print("Training session:", format_time(time.perf_counter() - session_started))
else:
    print("Full spatial training is off. Set RUN_SPATIAL_TRAINING=True when ready.")


## 8. Compare spatial models with the saved controls

Pending models have blank scores. Compare each new model with its matched 3-layer global-vector control. Attention also changes parameter counts; report these changes and timings rather than attributing any gain to depth. The test set remains reserved.

In [ ]:
comparison_rows = []
checked_parameters = {}
saved_check_path = PROJECT_DIR / "captioning_checks" / "spatial_verification.json"
if not saved_check_path.exists():
    saved_check_path = PROJECT_DIR / "spatial_verification.json"
if saved_check_path.exists():
    saved_check = json.loads(saved_check_path.read_text(encoding="utf-8"))
    if saved_check["implementation"] == SPATIAL_IMPLEMENTATION_ID:
        for check in saved_check["checks"]:
            if check["config"] == make_spatial_config(check["config"]["decoder"]):
                checked_parameters[check["config"]["run_name"]] = check["smoke"][
                    "trainable_parameters"
                ]
for decoder, control in CONTROL_RUNS.items():
    config = make_spatial_config(decoder)
    for label, folder, visual_input in [
        ("Global " + decoder.split("_")[0], GLOBAL_EXPERIMENT_DIR / control, "1 global vector"),
        (decoder, SPATIAL_EXPERIMENT_DIR / config["run_name"], "49 spatial locations"),
    ]:
        row = {
            "model": label,
            "input": visual_input,
            "layers": 3,
            "parameters": None,
            "best_epoch": None,
            "val_loss": None,
            "bleu1": None,
            "bleu4": None,
            "cider_d": None,
            "training_time": None,
            "device": None,
            "status": "pending",
        }
        if visual_input == "49 spatial locations":
            row["parameters"] = checked_parameters.get(config["run_name"])
        if (folder / "summary.json").exists():
            summary = json.loads((folder / "summary.json").read_text(encoding="utf-8"))
            for key in [
                "best_epoch",
                "val_loss",
                "bleu1",
                "bleu4",
                "cider_d",
                "training_time",
                "status",
            ]:
                row[key] = summary[key]
            row["parameters"] = summary["trainable_parameters"]
            row["device"] = "; ".join(item["name"] for item in summary["devices"])
        comparison_rows.append(row)

columns = list(comparison_rows[0])
lines = ["|" + "|".join(columns) + "|", "|" + "|".join(["---"] * len(columns)) + "|"]
for row in comparison_rows:
    values = [
        (
            ""
            if row[key] is None
            else f"{row[key]:.4f}" if isinstance(row[key], float) else str(row[key])
        )
        for key in columns
    ]
    lines.append("|" + "|".join(values) + "|")
display(Markdown("\n".join(lines)))
save_json(SPATIAL_EXPERIMENT_DIR / "comparison.json", comparison_rows)
with (SPATIAL_EXPERIMENT_DIR / "comparison.csv").open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=columns)
    writer.writeheader()
    writer.writerows(comparison_rows)
print(
    "Global experiments completed: 79 | spatial experiments completed:",
    sum(
        row["status"] == "finished"
        for row in comparison_rows
        if row["input"] == "49 spatial locations"
    ),
    "/ 2",
)
print("The global control scores are preserved. No labeled-test metric was used here.")


## 9. Visual locations and learned attention

The 7 x 7 grid describes CNN feature locations, not independent image crops: their receptive fields overlap. An activation heatmap is not learned decoder attention. After LSTM training, the second example uses the saved best model's attention for its own generated words. Attention weights alone are not a causal explanation.

In [ ]:
def display_photo(data, source, image_id):
    for root in [data["data_root"], Path.home() / "Downloads" / "MSCOCO" / "coco2017"]:
        path = root / source / f"{image_id:012d}.jpg"
        if path.exists():
            with Image.open(path) as original:
                photo = ImageOps.exif_transpose(original).convert("RGB")
                return ImageOps.pad(
                    photo,
                    (224, 224),
                    method=Image.Resampling.BILINEAR,
                    color=(124, 116, 104),
                    centering=(0.5, 0.5),
                )
    return None


if spatial_data is not None:
    preview = CachedCaptionDataset(spatial_data, "resnet50", "val", 42)
    try:
        fig, axes = plt.subplots(2, 2, figsize=(9, 8))
        for index in range(2):
            item = preview[index]
            photo = display_photo(data, "val2017", item["image_id"])
            if photo is not None:
                axes[index, 0].imshow(photo)
                axes[index, 0].set_xticks(np.arange(0, 225, 32))
                axes[index, 0].set_yticks(np.arange(0, 225, 32))
                axes[index, 0].grid(color="white", alpha=0.6)
                axes[index, 0].set_title(f"Image {item['image_id']}: padded 224 x 224")
            else:
                axes[index, 0].text(
                    0.1, 0.5, "Raw photo unavailable; training can still use the cache.", wrap=True
                )
                axes[index, 0].axis("off")
            strength = item["features"].norm(dim=1).reshape(7, 7).numpy()
            heatmap = axes[index, 1].imshow(strength, cmap="viridis")
            axes[index, 1].set_title("CNN activation magnitude (not attention)")
            fig.colorbar(heatmap, ax=axes[index, 1], fraction=0.046)
            print("Image", item["image_id"], "| reference:", item["references"][0])
        plt.tight_layout()
        plt.savefig(SPATIAL_EXPERIMENT_DIR / "spatial_feature_examples.png", dpi=140)
        plt.show()

        lstm_config = make_spatial_config("lstm_attention")
        folder = SPATIAL_EXPERIMENT_DIR / lstm_config["run_name"]
        if (folder / "best_weights.pt").exists():
            identity = json.loads((folder / "config.json").read_text(encoding="utf-8"))
            assert identity == experiment_identity(spatial_data, lstm_config, evaluator)
            attention_model = build_model(spatial_data, lstm_config, device)
            attention_model.load_state_dict(
                torch.load(folder / "best_weights.pt", map_location=device, weights_only=True)
            )
            attention_model.eval()
            item = preview[0]
            photo = display_photo(data, "val2017", item["image_id"])
            with torch.no_grad():
                features = item["features"].unsqueeze(0).to(device)
                generated = attention_model.greedy(
                    features, data["ids"]["start"], data["ids"]["end"], 33
                )[0]
                print("Spatial LSTM:", tokens_to_text(generated, data["id_to_token"]))
                if generated and photo is not None:
                    inputs = torch.tensor([[data["ids"]["start"]] + generated[:-1]], device=device)
                    _, weights = attention_model.forward_with_attention(features, inputs)
                    maps = weights[0].float().cpu().reshape(-1, 7, 7).numpy()
                    shown = min(8, len(generated))
                    fig, axes = plt.subplots(2, 4, figsize=(12, 6))
                    for index, ax in enumerate(axes.flat):
                        ax.axis("off")
                        if index < shown:
                            ax.imshow(photo)
                            ax.imshow(
                                maps[index],
                                extent=(0, 224, 224, 0),
                                cmap="magma",
                                alpha=0.55,
                                interpolation="bilinear",
                                vmin=0,
                                vmax=maps[:shown].max(),
                            )
                            ax.set_title(data["id_to_token"][generated[index]])
                    fig.suptitle(
                        "Best spatial LSTM: attention for its generated words (shared scale)"
                    )
                    plt.tight_layout()
                    plt.savefig(SPATIAL_EXPERIMENT_DIR / "lstm_generated_attention.png", dpi=140)
                    plt.show()
            del attention_model
        else:
            print("Learned attention maps become available after spatial LSTM training.")
    finally:
        preview.close()
else:
    print("Complete the spatial cache to display the 49-location examples.")


## 10. Verification record and later merge

This notebook is self-contained. For the final submission, append these sections after the original results, keep one imports/settings section, and retain distinct model names and result groups. The saved report distinguishes development checks from full training. Never load a global-vector checkpoint into a spatial decoder.

In [ ]:
verification_path = PROJECT_DIR / "captioning_checks" / "spatial_verification.json"
if not verification_path.exists():
    verification_path = PROJECT_DIR / "spatial_verification.json"
if verification_path.exists():
    report = json.loads(verification_path.read_text(encoding="utf-8"))
    assert report["implementation"] == SPATIAL_IMPLEMENTATION_ID
    print("Development checks passed:", report["passed"], "| device:", report["device"]["name"])
    print(
        "Full spatial experiments trained by these checks:",
        report["full_spatial_experiments_trained"],
    )
else:
    print("No saved spatial development report. Preflight remains required before training.")
